# Databricks GenAI Engineering Pathway — Certification Crash Course

**This is the single, comprehensive study notebook.** It merges every concept, function and code
pattern from **all five courses** of the *Generative AI Engineering Pathway* — both the official
lecture pages (the `.mhtml` files in `W2`–`W5`) and the hands-on lab notebooks (`W1`–`W4`) —
plus everything the practice question banks (`Questions/`) tested that the course material
under-covered (e.g. `ai_prep_search`, scorer lifecycle management, UC trace grants).
You do not need to run these cells — read top to bottom.

### Course → section map
The `W#` folders are **offset**: each folder holds one course's *labs* and the next folder holds
that course's *lecture pages*. Use this table, not the folder numbers.

| Course (official name) | Section here | Labs | Lecture pages |
|---|---|---|---|
| Generative AI Fundamentals | **§0** | — | — (question bank only) |
| Building RAG Agents with Agent Bricks | **§1** | `W1` | `W2` |
| Building Agentic Applications on Databricks | **§2** | `W2` | `W3` |
| Agent Evaluation on Databricks | **§3** | `W3` | `W4` |
| Deploying and Monitoring Agent Applications on Databricks | **§4** | `W4` | `W5` |

### How this is organized (the order concepts are actually needed)
0. **Generative AI Fundamentals** — tokens, context windows, grounding, RAG rationale, model classes, LLM-as-judge
1. **Building RAG Agents with Agent Bricks** — context engineering, `ai_parse_document`/`ai_classify`/`ai_extract`/`ai_prep_search`, chunking, embeddings, AI Search, MLflow basics, Knowledge Assistant
2. **Building Agentic Applications** — agents & tools, single vs multi-agent, UC Functions, LangChain, OpenAI Agents SDK, MLflow tracing, `ResponsesAgent`, Agent Bricks & Genie, MCP, Unity AI Gateway
3. **Agent Evaluation** — why traditional testing breaks, `mlflow.genai.evaluate()`, the judge spectrum, guideline judges, custom judges & scorers, `Feedback`, multi-turn judges, offline vs online, sampling, scorer lifecycle, backfill, archival
4. **Deploying & Monitoring** — Model Registry, batch & real-time serving, Provisioned Throughput, A/B testing, inference tables & Lakehouse Monitoring, and (§4.10) the four-stage agent lifecycle + Databricks Apps / DABs
5. **Exam traps & gotchas** cheat sheet
6. **Full API / function alphabetical index**

> Anything marked **⚠** is a high-frequency trap, or a point the question banks tested more
> precisely than the lecture text stated it.

> **Coverage caveat**: the published Databricks exam guide is not reachable from this machine, so
> completeness here is verified against **100% of the supplied lecture pages, lab notebooks and
> question banks** — not against a published blueprint.

---
# 0. Generative AI Fundamentals ⚠

These are the baseline concepts the whole pathway assumes you already know. They are tested by the
*Generative AI Fundamentals* question bank but are **not** spelled out in any lab notebook or
lecture page — so this section is gap-fill.

### 0.1 Tokens, parameters, and context windows
- **Tokens**: the units (word pieces) an LLM reads and writes. Cost, latency and context limits are all measured in tokens, not characters or words.
- **Parameters**: the model's **internal weights/settings that define its structure and learned "intelligence"** — not documents, not tokens, not output length.
- **Context window**: the model's total working-memory limit (input + output tokens combined). When a conversation **exceeds** it, the model **drops the earliest information** to make room for new input — this silent truncation is a direct cause of **hallucination** (the model "forgets" facts it was told earlier and fills the gap with plausible-sounding invention).

### 0.2 Why RAG improves on a "vanilla" LLM
A vanilla LLM only knows what was in its training data, frozen at a cutoff date. **Retrieval Augmented Generation (RAG) lets the model look up real-time, trusted external information before generating an answer**, instead of relying purely on frozen training weights. This is why RAG defeats knowledge-cutoff and hallucination problems that no amount of prompt engineering can fix (see §1.1).

### 0.3 Grounding
**Grounding** = anchoring a model's responses in **specific, verified organizational data**, so answers are traceable to real sources instead of invented. Grounding is the mechanism that makes RAG/agent answers trustworthy in an enterprise setting — it is not about limiting creativity or hosting location, it is about evidentiary backing for claims.

### 0.4 The "Brilliant Intern" analogy
LLMs behave like a **brilliant intern**: extremely knowledgeable and articulate, but **takes instructions extremely literally and has no inherent business context**. It will confidently do exactly what you *say*, not what you *meant*, and knows nothing about your company's specific policies/data unless you tell it (→ this is the argument for context engineering and RAG, not for vague prompting).

### 0.5 GenAI vs. AI Agent — the fundamental distinction
- **GenAI (standalone model)**: single-step generation — prompt in, completion out. No autonomous multi-step behavior.
- **AI Agent**: uses **reasoning to drive multi-step, adaptive workflows** — it plans, calls tools, observes results, and adjusts its own next action, rather than answering once and stopping.

### 0.6 Model classes — pick by task shape
| Model class | Best for |
|---|---|
| **Small Language Models (SLMs)** | High-volume, repetitive tasks where **low latency/speed matters more than deep reasoning** (e.g. classification, batch scoring) |
| **Large Language Models (LLMs)** | General-purpose reasoning and generation at moderate scale |
| **Reasoning-focused models** | Problems needing extended internal chain-of-thought before answering (slower, costlier) |
| **Frontier models** | Maximum capability ceiling, highest cost/latency — reserve for the hardest problems |

### 0.7 LLM-as-Judge — pros and a documented con
LLM judges let you evaluate free-text, non-deterministic outputs at scale without a human in the loop for every example. A well-documented **con: "verbosity bias"** — a judge model **may favor longer responses regardless of whether they are actually more accurate or higher quality**. (Other known judge biases worth knowing for the exam: position bias — favoring whichever answer appears first/second in a pairwise comparison — and self-preference bias — a judge favoring outputs that resemble its own model family's style.)

### 0.8 What actually separates GenAI leaders from the pack
Not raw compute, not "the newest frontier model" — the primary differentiator is **connecting GenAI to your organization's own unique, proprietary data and domain expertise**. Anyone can call a frontier model API; the moat is what you ground it in.

### 0.9 Agent Bricks — one-line role reminder
The **Supervisor Agent** (Multi-Agent Supervisor) is the component that acts as an **intelligent router**, directing a user request to the correct specialized sub-agent or tool. (Full detail in §2.10.)

---
# 1. Building RAG Agents with Agent Bricks

## 1.1 Prompt Engineering vs. Context Engineering
- **Prompt Engineering**: tactical — refining the instruction text itself. Its scope is **system instructions, few-shot examples and the user prompt**. Cannot fix frozen training data, hallucination or ambiguity — these are **hard boundaries** of prompting.
- **Context Engineering**: architectural — designing the **entire input environment** the model sees. It includes everything prompt engineering does **plus** retrieved documents and their metadata, conversation history and user constraints (persisted in **Lakebase**), and tool usage (via **MCP** and **Genie**).
- **Analogy**: prompt engineering is writing a good exam question; context engineering is designing the whole exam room — what reference material is on the desk, what the candidate remembers, and what tools they may use.
- **Reasoning vs non-reasoning models**: non-reasoning models (Llama 3, GPT-4o) need explicit Chain-of-Thought prompting to break down logic. Reasoning models (OpenAI o1-class) generate their own internal chain of thought — manual CoT prompting on them is redundant/counterproductive; context engineering for them focuses on goal + constraints, not the thinking steps.

**The three LLM limitations RAG exists to fix** (none are solvable by prompting alone):
1. **Knowledge cutoffs** — no visibility past the training-data date.
2. **Hallucination** — fabricates plausible-sounding facts/citations when it lacks real references.
3. **Missing private context / ambiguity** — no access to your proprietary data, so it defaults to the generic interpretation of a term (e.g. "secure a lakehouse" → physical security, not Databricks governance).

## 1.2 RAG — Retrieve, Augment, Generate (in that order)
**RAG is the architectural pattern; a "Retrieval Agent" is a concrete implementation of it** (adds query routing, retrieval orchestration, context assembly). The three RAG stages, **in order**:
1. **Retrieve** — search a knowledge base (a Databricks AI Search index) for relevant chunks.
2. **Augment** — inject those chunks into the model's context window.
3. **Generate** — the model answers using *only* the injected context.

## 1.3 Context Rot — the two failure patterns of naive RAG
- **Context poisoning**: dumping irrelevant/conflicting chunks into the prompt confuses the model.
- **Lost in the middle**: models over-weight the very start/end of a long context and **overlook information buried in the middle** — fix with filtering, reranking, and keeping context tight (this is also a core argument for smaller, focused chunks — §1.11).

## 1.4 Context engineering techniques
- **System prompt design**: explicit role definition, negative constraints ("do not..."), enforced output format (JSON/YAML/Markdown) for deterministic downstream parsing.
- **Strict grounding**: instruct the model to answer *only* from provided context ("If the answer is not present, say you don't have that information") + **metadata filtering** (filter chunks by e.g. `year=2024` *before* the model ever sees them — Unity Catalog metadata is the filter mechanism).
- **Multi-turn state management**: summarization of history, moving-window truncation, selective persistence (keep user name/project ID, discard the rest).
- **Token economics**: naive RAG pulling 50 documents per query burns tokens fast. Fixes: **just-in-time retrieval** (agent has a tool to fetch a specific section only when actually asked, instead of loading everything up front) and **reranking** (score top ~50 candidates, inject only the top 3–5 into the final context).

## 1.5 Data architecture: Volumes → Delta tables
- **Unity Catalog Volumes** govern non-tabular files (PDFs, images) the same way UC governs tables — raw files are "Bronze". Path shape: `/Volumes/catalog/schema/volume_name/file.pdf`.
- **Delta Lake tables** store the parsed/chunked *text* ("Silver/Gold") — ACID + versioning.

### The document-processing pipeline — three stages
| Stage | Function(s) | Output |
|---|---|---|
| **1. Parse** | `ai_parse_document` | A **VARIANT** per document containing text, layout, tables, images, page structure |
| **2. Classify & extract** | `ai_classify`, `ai_extract` | Document type labels and structured fields, derived **from the parsed VARIANT** |
| **3. Chunk** | `ai_prep_search` | Retrieval-ready chunks with context-enriched and clean text variants |

⚠ Stages 2 and 3 **both read the parsed VARIANT and can run in parallel** — classification and extraction are not prerequisites for chunking. Their results are then **joined onto the chunk table as metadata columns**, which is what lets you filter and rank by document type or extracted field at query time.
Full flow: Volume (raw files) → `ai_parse_document` → clean/transform → `ai_prep_search` chunks (+ classify/extract metadata joined on) → embed → AI Search index.

## 1.6 `ai_parse_document` — the core parsing function
Native Databricks AI function, serverless, no separate infra. Uses OCR + multimodal LLMs so it "sees" layout (multi-column order, tables spanning pages, captions tied to their image) instead of naive text extraction.

```sql
SELECT path, ai_parse_document(content, map('version', '2.0')) AS parsed_doc
FROM read_files('/Volumes/catalog/schema/docs/', format => 'binaryFile')
```
```python
from pyspark.sql.functions import expr
parsed_df = docs_df.withColumn(
    "parsed_content",
    expr("ai_parse_document(content, map('version','2.0','imageOutputPath','/Volumes/.../images/'))")
)
```
**Parameters:** `content` (BINARY, required) · `version` · `imageOutputPath` (where extracted images go) · `descriptionElementTypes` (`*`/`table`/`image`/`text`).
⚠ **The `version` flag matters**: **`'2.0'` is the recommended version** — it improves layout handling and adds HTML-formatted tables, and the v2 `ai_extract` / `ai_classify` examples are tuned for its output shape. Omit it and you get the older schema.
**Returns VARIANT** with `document.pages[]` (page_number, text, tables, images), `document.elements[]` (id, type, content, bbox coordinates), `document.metadata`, `error_status`, `corrupted_data`. **Schema v2.0 capabilities:** layout awareness, auto figure/chart descriptions, bounding boxes for UI source-highlighting.
**Gotcha:** always check `error_status` / verify sample output — it can fail silently on some files.

## 1.7 Turning the parsed VARIANT into rows — `variant_explode` ⚠ GAP-FILL
`ai_parse_document` returns one VARIANT per document containing an **array** of elements. To filter/classify each element individually you need **one row per element**:
```sql
SELECT parsed_docs.path, element.value:content::STRING AS content
FROM parsed_docs,
  LATERAL variant_explode(parsed_docs.parsed:document:elements) AS element
WHERE element.value:content IS NOT NULL
```
`variant_explode` converts a VARIANT array/object into rows, exposing exactly **three columns**:
| Column | Type | Meaning |
|---|---|---|
| `pos` | `INT` | Zero-based position within the array/object |
| `key` | `STRING` | Object key — **`NULL` when the input is an array** |
| `value` | `VARIANT` | The element value itself |

- Used with `LATERAL` (like a lateral `explode`, but VARIANT-aware). `posexplode` requires an already-materialized ARRAY column, not VARIANT, so it does **not** work on the parse output.
- ⚠ **NULL or non-VARIANT input produces *no rows at all*** — the parent row silently disappears. Use **`variant_explode_outer`** when you need to preserve rows whose VARIANT is null or empty.

## 1.8 `ai_classify` — label a block of text ⚠ GAP-FILL
Classifies text into a **fixed set of labels** you supply. Returns a VARIANT whose `response` field is an **array**; the top predicted label is `response[0]`:
```sql
SELECT element_id, ai_classify(content, array('policy','invoice','contract')) AS cls
FROM elements
-- top label:
SELECT element_id, cls:response[0] AS top_label FROM classified
```
Filter out noise before classifying — short items like page numbers/single-word headers produce noisy labels, so a length guard is standard practice: `WHERE LENGTH(content) > 50` for classification, a higher bar (e.g. `> 100`) for extraction since extraction needs more surrounding context to find fields reliably.

## 1.9 `ai_extract` — pull structured fields out of text
```sql
SELECT ai_extract(document_text, 'Extract invoice_date, vendor_name, and total_amount as JSON') AS fields
FROM invoices
```
Returns a VARIANT/JSON object with the requested fields. Can run **in the same `SELECT` alongside `ai_classify`** on the same parsed elements — both run in parallel per row; unpack `cls:response[0]` and `ext:response` into columns in one pass rather than separate jobs on separate clusters.

## 1.10 `ai_query` — call any Foundation Model endpoint from SQL/Python
```sql
SELECT ai_query('databricks-claude-sonnet-4-6', CONCAT('Summarize: ', document_text)) AS summary
FROM documents
```
Used for LLM-powered **semantic cleaning** of parsed output into clean markdown (vs. a fast/cheap plain-text-extraction UDF which is faster but loses structure). Also used for **batch inference** directly in SQL (§4.2) — no custom model code needed when using a Foundation Model API endpoint. Gotcha: Claude models return plain text by default — do not pass a `responseFormat` for them.

## 1.11 `ai_prep_search` — purpose-built chunking function ⚠ GAP-FILL
Takes the **full `ai_parse_document` VARIANT output for a document** (not a flattened single-text column — passing already-flattened element text defeats it, since it needs the original structure) and produces **semantic chunks enriched with document context** (titles, headers, page metadata folded in). Each output chunk carries **two different text variants**:
- **`chunk_to_embed`** — the context-enriched text (with titles/headers/metadata baked in) → send **this** to the embedding model, because the added context improves findability/recall.
- **`chunk_to_retrieve`** — the clean original chunk text with no injected metadata → show **this** to the LLM at generation time, because clean text reads better and is more accurate for the model to reason over.

So: **embed `chunk_to_embed`, retrieve/display `chunk_to_retrieve`.** When creating the Delta Sync index, `embedding_source_column` must point at the *_embed* column — pointing it at the raw retrieval column under-performs on findability.

### Chunking principles (apply whichever chunker you use)
| Principle | Rule |
|---|---|
| **Fit the embedding model** | Chunks must fit the embedding model's context window (**e.g. 512 tokens**). Anything beyond is **silently truncated** — no error, just lost text and an incomplete vector |
| **Overlap** | Keep a **10–20% overlap** between adjacent chunks so information spanning a boundary isn't split away. Zero overlap loses boundary context |
| **Granularity trade-off** | Smaller chunks → higher precision, more chunks to manage. Larger chunks → more context per hit, more noise and dilution |
| **"Lost in the middle"** | Smaller, focused chunks reduce the risk that key information sits in the ignored middle of a long context (§1.3) |

## 1.12 Chunking strategies (when not using `ai_prep_search`)
- **Fixed-size** (legacy): cheap, predictable, but splits sentences/paragraphs mid-thought.
- **Recursive / semantic** (recommended default): splits on logical linguistic boundaries.
- **Embedding-based semantic chunking**: uses an embedding model to detect topic-similarity breakpoints — only splits when similarity drops below a threshold; each chunk = one coherent concept; more expensive, higher quality.
- **Windowed summarization**: each chunk carries a short summary of the preceding chunks for broader context without full re-embedding cost.
- **LangChain tool**: `RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200, separators=[...])` — apply via Spark `mapInPandas` for distributed chunking.
- **Rule of thumb**: max chunk size ≈ 1/4–1/3 of the embedding model's context window; exceeding the model's token limit **silently truncates** the excess.

## 1.13 Embeddings fundamentals
- **Embedding**: a numerical vector representation of content; similar meaning → mathematically close vectors. "cozy apartment near the park" and "comfortable flat close to green space" embed close together despite sharing no words.
- **Embedding alignment**: you **must use the same embedding model** for indexing documents AND encoding queries, or the two live in misaligned vector spaces and retrieval quality collapses.
- **Respect the context window**: embedding models have token limits (e.g. **512 tokens**); text beyond the limit is **silently truncated** — no error.
- **Dimensionality trade-off**: 384 / 768 / 1024+ — higher captures more nuance but costs more to store and search.

**How similarity search runs**: embed the query → find nearest vectors in the index (cosine similarity or another metric) → return the corresponding chunks ranked by score.

### Search methods
| Method | Best for |
|---|---|
| **Similarity/semantic search (ANN)** | Natural-language queries, synonym-tolerant, conceptual matches |
| **Full-text/keyword search** | Exact terms — part numbers, product codes, proper nouns, standard codes (e.g. "ISO 13849-1") |
| **Hybrid search** | Domain content where *both* semantics and exact keywords matter — usually the best overall accuracy |

### Distance/similarity metrics
- **Euclidean (L2)** / **Manhattan (L1)** — distance metrics, lower = more similar; used for clustering/outlier detection.
- **Cosine similarity** — the standard for text embeddings; measures the **angle** between vectors, not magnitude, so it is robust to document-length differences. Small angle → score near **1.0**; large angle → score toward **0**.

### Exact vs approximate search
- **KNN (exact)**: compares against every vector — accurate but doesn't scale.
- **ANN (approximate, e.g. HNSW, FAISS)**: Databricks AI Search's default — trades a little precision for dramatic speed/scale gains (millions+ vectors).

### Reranking
Similarity (vector closeness) ≠ actual relevance. Two-stage pattern: (1) broad ANN retrieval of top 20–50 candidates, (2) a cross-encoder reranker re-scores/reorders them for true relevance, (3) only the reordered top few go to the LLM.
```python
from databricks.vector_search.reranker import DatabricksReranker
results = index.similarity_search(
    query_text=q, columns=["path","chunk"], num_results=5,
    reranker=DatabricksReranker(columns_to_rerank=["chunk"])
)
```
Trade-off: better accuracy, less hallucination — at the cost of extra latency/inference cost. Use selectively for high-value queries.

## 1.14 Databricks AI Search — the vector database
Integrated vector DB in the Lakehouse (formerly *Databricks Vector Search*) — no separate infra; governed by Unity Catalog.

**Three ingestion modes:**
1. **Managed embeddings (Delta Sync)** — recommended default: point at a Delta table, Databricks computes embeddings automatically via a Model Serving endpoint, index auto-syncs on table changes.
2. **Self-managed embeddings (Delta Sync)** — you compute embeddings yourself into a Delta column; AI Search still auto-syncs the index, but you own the embedding pipeline.
3. **Direct CRUD API** — insert/update/delete vectors directly via REST/SDK, no Delta table backing — real-time/custom workflows.

### What happens when an index is created
1. Read the source Delta table → 2. Compute embeddings (calls the embedding model on each row of the embedding source column) → 3. Build the ANN index for fast nearest-neighbour lookup → 4. Store the synced columns alongside the vectors so they can be returned in results. The **initial sync can take several minutes**.

**Index creation parameters (know these by name):**
| Parameter | What it does |
|---|---|
| `endpoint_name` | Which AI Search endpoint serves the index |
| `source_table_name` | The Delta table containing your chunks |
| `index_name` | Fully-qualified UC name for the index |
| `pipeline_type` | `TRIGGERED` (sync on demand) or `CONTINUOUS` (sync automatically) |
| `primary_key` | Unique row identifier used to track which rows have been processed |
| `embedding_source_column` | Which column is sent to the embedding model |
| `embedding_model_endpoint_name` | The model converting text → vectors (e.g. `databricks-gte-large-en`) |
| `columns_to_sync` (a.k.a. `columns`) | Which columns to store alongside vectors so they can be returned in results |
| `embedding_dimension` / `embedding_vector_column` | Required for **self-managed** embeddings instead of the two `embedding_*` params above |

```python
from databricks.vector_search.client import VectorSearchClient
vsc = VectorSearchClient(disable_notice=True)

vsc.create_endpoint(name=vector_search_endpoint, endpoint_type="STANDARD")
vsc.list_endpoints()                       # -> {"endpoints": [...]}
vsc.get_endpoint(name=vector_search_endpoint)

vsc.create_delta_sync_index_and_wait(
    endpoint_name=vector_search_endpoint,
    index_name=f"{catalog}.{schema}.my_index",
    source_table_name=source_table,
    primary_key="id",
    embedding_source_column="chunk_to_embed",          # see §1.11
    embedding_model_endpoint_name="databricks-gte-large-en",
    pipeline_type="TRIGGERED",                          # or "CONTINUOUS"
    columns_to_sync=["id", "chunk_to_retrieve", "path"],
)

index = vsc.get_index(index_name=index_name)
index.describe()["status"]["ready"]        # poll; status.state ∈ ONLINE / PROVISIONING
index.sync()                               # manual sync for TRIGGERED pipelines
```

**Hard requirement — Change Data Feed**: the source Delta table **must have CDF enabled** for Delta Sync on standard endpoints, and it must be set **before** creating the index:
```sql
ALTER TABLE chunks_table SET TBLPROPERTIES (delta.enableChangeDataFeed = true)
```
With CDF on, AI Search processes only rows **changed since the last sync** instead of rebuilding. Forgetting this is the textbook cause of "incremental updates fail to propagate" — the fix is enabling CDF, **not** switching `query_type` to `FULL_TEXT`, **not** raising `num_results`, and **not** moving the table into a UC Volume (indexes sync from Delta **tables**, not Volumes).

**Endpoint types**: Standard (general purpose, continuous sync available, `endpoint_type="STANDARD"`) vs. Storage-Optimized / `"PROVISIONED"` (very large indexes, dedicated compute, data isolation, triggered sync only, higher cost).

### Querying — `similarity_search()` and its filter syntax
```python
results = index.similarity_search(
    query_text="pet policy", columns=["path", "chunk_to_retrieve"], num_results=5,
    query_type="hybrid",                                # "ANN" | "hybrid" | "FULL_TEXT"
    filters={"path LIKE": "doc.pdf"},
)

filters={"path NOT": "archive/old.pdf"}
filters={"page_number >": 10, "page_number <": 50}
filters={"year": 2024}                                  # bare key = equality
```
Filters are a **dict of `"column OP"` keys**. String filters match **whitespace-separated tokens**, not arbitrary substrings. `query_type` accepts `"ANN"` (default semantic), `"HYBRID"`, and `"FULL_TEXT"` (exact keyword only — beta, must be enabled in the workspace).

### Governance across the retrieval pipeline
UC Volume (source PDFs) → Delta table (chunks + metadata) → AI Search index — all three governed by **Unity Catalog privileges**. The **AI Search endpoint** is governed separately by **endpoint ACLs**. Audit visibility comes from system tables; lineage is captured across the UC tables/paths in the pipeline.

### AI Search best practices (course checklist)
1. Minimize embedding dimensionality — pick the **lowest** dimension that preserves quality.
2. Keep `num_results` moderate (**10–100**, not thousands).
3. Choose the right endpoint SKU (Standard vs Storage-Optimized).
4. Use filters + metadata to narrow scope before ranking.
5. Prefer ANN for speed; use hybrid when exact keywords matter.
6. Enable CDF on the source table.
7. Test multiple chunk sizes and embedding models.
8. Monitor index refresh latency.

### AI Playground — test retrieval before writing agent code
Catalog Explorer → select the AI Search index → **"Try in Playground"**. This pre-wires the index as a retrieval tool and lets you pick an LLM, so you can validate retrieval quality (and catch chunking/embedding problems) before building any agent. The same Playground prototypes UC-function tool-calling agents in Module 2, and its **Get code → Export to Databricks Apps** action scaffolds an OpenAI-SDK app project (§4.10).

### LangChain RAG-chain components
```python
from databricks_langchain import DatabricksVectorSearch, ChatDatabricks
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

vs = DatabricksVectorSearch(index, columns=["id", "content", "pdf_name"])
retriever = vs.as_retriever(search_kwargs={"k": 3})
llm = ChatDatabricks(endpoint="databricks-gpt-5-1", temperature=0.1, max_tokens=250)
doc_chain = create_stuff_documents_chain(llm, prompt)   # "stuff" = concatenate all docs into the prompt
chain = create_retrieval_chain(retriever, doc_chain)     # retriever + doc chain = full RAG chain
```

### Alternative chunker — `SentenceSplitter` (llama_index)
```python
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.utils import set_global_tokenizer
from transformers import AutoTokenizer

set_global_tokenizer(AutoTokenizer.from_pretrained("hf-internal-testing/llama-tokenizer"))
splitter = SentenceSplitter(chunk_size=500, chunk_overlap=50)
nodes = splitter.get_nodes_from_documents([Document(text=doc)])   # node.text = the chunk
```
`set_global_tokenizer` matters because `chunk_size` is counted in **tokens**, so the splitter must use the same tokenizer as your model.

## 1.15 MLflow essentials for retrieval agents
- **Experiment** = logical project container. **Run** = one specific configuration (system prompt, chunk size, temperature, retriever settings) captured for comparison/reproducibility.
```python
import mlflow
mlflow.set_experiment("my_retrieval_agent")
with mlflow.start_run(run_name="baseline_v1"):
    mlflow.log_param("chunk_size", 2000); mlflow.log_metric("retrieval_quality", 0.85)
```
- **UC Model Registry**: `mlflow.set_registry_uri("databricks-uc")`, three-level namespace `catalog.schema.model_name`, `mlflow.register_model(...)`, `client.set_registered_model_alias(name=..., alias="Champion", version=...)`.
- **Model flavors**: `mlflow.langchain.log_model(...)` (auto-handles chain serialization) vs. `mlflow.pyfunc.log_model(python_model=MyModel())` (custom `predict(context, model_input)` logic — needed when native flavors can't express custom reranking/routing).
- **Tracing**: `mlflow.langchain.autolog()` for automatic tracing, or manual `@mlflow.trace` / `with mlflow.start_active_span(...)`. **Trace** = the full request lifecycle; **Span** = one step in it. Diagnosing failures: empty/irrelevant retriever span output → embedding/chunking problem; slow `vector_search` span vs fast `llm_generation` span → DB latency not model latency; retriever span correct but LLM output ignores it → prompt/grounding failure, not a retrieval failure.

### Agent-as-Code pattern (`agent.py` + `config.yaml`)
Rather than logging an in-memory object, log the agent as a **source file**, which makes it portable and re-loadable at serving time:
- `agent.py` — builds the agent, wraps it in a `ResponsesAgent` subclass, and ends with `mlflow.models.set_model(AGENT)` so MLflow knows which object is the model.
- `agent-config.yaml` — holds environment-specific values (`llm_endpoint_name`, `vector_search.index_name`, `vector_search.num_results`) that `agent.py` reads via a `_load_config()` helper.
- When logging: pass `python_model="agent.py"` and include the YAML via `code_paths=["agent-config.yaml"]` (or `artifacts={...}`), so the same code runs in dev and prod with a different config file.
```python
# end of agent.py
AGENT = MyResponsesAgent(agent)
mlflow.models.set_model(AGENT)
```

## 1.16 Agent Bricks: Knowledge Assistant (managed RAG)

### Managed RAG vs Custom RAG
| | **Managed RAG (Knowledge Assistant)** | **Custom RAG (Agent Framework + AI Search)** |
|---|---|---|
| You provide | Documents (UC Volume, table, or an existing index) | The whole pipeline, built yourself |
| System handles | Parsing, chunking, embedding, indexing, serving | Nothing — you own each step |
| Optimization | Automatic via ALHF feedback loops | Manual tuning and experimentation |
| Setup time | Minutes | Hours to days |
| Customizability | Configuration- and feedback-driven | Essentially unlimited |
| Best for | Standard doc Q&A where speed to production matters | Unique requirements, complex multi-step reasoning, novel architectures |

### The four stages of a Knowledge Assistant pipeline
Two offline, two online, plus a feedback loop:
1. **Indexing (offline)** — ingest UC Volumes/tables (or an existing index) → parse → chunk → embed (`databricks-gte-large-en`) → build the AI Search index, normalizing metadata.
2. **Retrieval (online)** — receive the user query → the **Instructed Retriever** plans retrieval across the configured sources into tailored sub-queries → search and rank (AI Search → rerank → evidence set).
3. **Generation (online)** — an LLM grounds on the retrieved chunks and produces an answer with **doc- and page-level citations** back to the sources (citations serve users and evaluators, not just testing).
4. **Feedback loop** — MLflow evaluation (tracing, LLM judges, task-specific eval datasets/guidelines) is the **evaluation layer**; **ALHF** is the **learning/optimization layer** that refines both retrieval behaviour and answer generation over time.

### Knowledge source types (up to **10 per agent**)
| Source type | Requirements | Best for |
|---|---|---|
| **Files in a UC Volume** | `txt`, `pdf`, `md`, `ppt/pptx`, `doc/docx`. Files **>50 MB are automatically skipped**. KA parses/chunks/embeds/indexes for you | Document collections that change over time (policies, manuals, specs) |
| **AI Search index** | A pre-built index that **must use `databricks-gte-large-en`** as its embedding model; you own the parsing/chunking/indexing | Custom RAG pipelines where you already manage index and schema |
| **Files in a UC table (file table)** | Must be a **streaming table OR have Change Data Feed enabled**, with a `content` column (**BINARY or STRING**) plus a `_metadata`/`metadata` struct (path, name, size, mod time) | Documents from connectors (SharePoint, Google Drive, Jira, Confluence) landing as file tables |

⚠ A file table does **not** need Parquet date-partitioning, does **not** need converting to a Volume, and does **not** need a precomputed embedding column — KA computes embeddings itself.

**Deployment**: creating a Knowledge Assistant auto-provisions *both* an AI Search endpoint and a Model Serving endpoint — both cost money and both are removed when the agent is deleted.

### Declarative vs. Code-First
| | **Declarative (Knowledge Assistant)** | **Code-First (Custom Agents / Supervisor)** |
|---|---|---|
| Approach | Declare *what* you want; the system learns and optimizes *how* | Write retrieval logic, prompts, tools and orchestration yourself |
| Setup time | Minutes | Hours to days |
| Customizability | Configuration- and feedback-driven | Essentially unlimited |
| Optimization | Automatic via ALHF and research upgrades | Manual tuning and experimentation |
| Who owns config | The platform | The developer |
| Redeploy for improvements | No — the quality loop updates in place | Yes |
| Best for | High-quality Q&A over docs, fast prototyping | Novel architectures, complex multi-step / tool-heavy workflows |

---
# 2. Module 2 — Building Agentic Applications (W3)

## 2.0 Platform requirements & the AI Bridge packages
- **Runtime**: Serverless compute, or Databricks Runtime **13.3 LTS+**; Python **3.10+**.
- **Core packages**: `databricks-agents` **1.2.0+**, `mlflow` **3.1.3+**. Install with `%pip install -U -qqqq databricks-agents mlflow` then `dbutils.library.restartPython()`.
- **AI Bridge** = the family of thin integration packages that connect Databricks resources to each agent framework: **`databricks-langchain`**, **`databricks-openai`**, **`databricks-dspy`**, all on top of **`databricks-ai-bridge`**. This is where `ChatDatabricks`, `UCFunctionToolkit`, `VectorSearchRetrieverTool`, `DatabricksVectorSearch` come from — pick the package matching your framework.

## 2.1 What is an AI agent
An **AI agent** is an autonomous software system that perceives its environment, makes decisions, and takes actions to reach a goal. Unlike traditional AI that needs continuous user input, an agent can **reason** about problems, **plan** sequences of actions, **adapt** to new information, **interact** with external systems, and **learn** from experience.
Core components: **LLM brain** (reasoning), **memory** (conversation/context state), **planning module** (breaks requests into steps), **tool interface** (external systems/APIs/functions), **execution engine** (runs planned actions, handles responses).
Complexity ladder: Simple Reflex → Model-Based Reflex → Goal-Based → Utility-Based → Learning agents.
**Not all LLMs can call tools** — tool-calling must be an explicit model capability, plus correct framework integration (LangChain/OpenAI SDK/DSPy) and agent-runtime setup (ResponsesAgent/LangGraph).

### Supervisor–worker (router–delegate) multi-agent architecture
A **Supervisor Agent** receives the request and analyses intent → delegates to the appropriate **worker** (Data Agent with UC functions/SQL, Research Agent with AI Search, General Agent for knowledge/reasoning) → the worker executes with its specialised tools → the supervisor synthesises and returns the response.

### Single agent vs. multi-agent — the decision table
| Factor | Single agent | Multi-agent |
|---|---|---|
| **Number of tools** | Best when the total is modest (**≈ up to 8–10**); beyond that tool selection and prompt management degrade (heuristic, not a hard limit) | Many tools spanning distinct capabilities (often **>10**) needing clear separation |
| **Domain complexity** | One focused or tightly related domain | Multiple distinct domains or verticals |
| **Instruction length** | Short, coherent system prompt one agent can follow | Would need a very long, fragmented or conflicting prompt in one agent |
| **Failure isolation** | Failures acceptable as one unit | Need to isolate, debug and replace individual capabilities |
| **Team ownership** | One team owns most behaviours | Different teams own different domains and evolve independently |
| **Latency** | Minimal overhead, fewer model calls | Extra routing/orchestration calls add latency and token cost |
| **Scalability / parallelism** | Fine while domain and tools stay bounded | Better for parallelising across domains and distributing context |

> **Start with a single agent.** Decompose only when you observe tool-selection errors, instruction conflicts, or when different teams need to own capabilities independently.

## 2.2 Three approaches to building agent tools on Databricks
1. **Unity Catalog Function Tools** (primary/recommended) — governed, discoverable, secured SQL/Python UDFs.
2. **Agent-code tools** — defined directly in agent code (REST calls, arbitrary/low-latency logic, or the SDK's `@function_tool`). Trade-off: no UC governance/discoverability.
3. **MCP tools** — standardized, interoperable tool format via Databricks-managed or custom MCP servers (§2.11).

**Common tool patterns:**
| Pattern | Description |
|---|---|
| **Structured data retrieval** | Query SQL tables, databases and structured sources (UC function or Genie space) |
| **Unstructured data retrieval** | Search document collections / perform RAG (AI Search) |
| **Code interpreter** | Let the agent run Python for calculations, data analysis and dynamic processing — an **execution sandbox**, not a lookup |
| **External connection** | Call outside services and APIs (e.g. Slack) |
| **AI Playground prototyping** | Attach UC tools to an agent in the Playground to prototype behaviour before writing code |

**Genie space** (natural-language analytics over structured data): a curated NL interface to governed Databricks data, where you define the data context and instructions Genie uses. A **Genie agent** is a Genie Space used as a specialised worker. Two integration paths: **Genie via managed MCP** (an external agent uses Genie as a tool) and **Genie as a subagent** inside a Supervisor Agent.
Routing split to memorize: **Genie space** → open-ended NL→SQL analytics · **AI Search index** → unstructured document retrieval · **UC function** → fixed, known, parameterized lookup · **Knowledge Assistant** → packaged document Q&A.

## 2.3 UC Functions as tools — SQL vs Python
| | SQL | Python |
|---|---|---|
| Registration | `CREATE OR REPLACE FUNCTION` | `DatabricksFunctionClient().create_python_function()` |
| Execution | Serverless only | Serverless (prod) or local (dev) |
| Typing | SQL data types | Python type hints **required** (`int/str/float/bool/list/dict`) |
| Docs | `COMMENT` clauses | Google-style docstring |
| Best for | Retrieval, aggregation, filtering, calcs | Business logic, external APIs, complex transforms |

```sql
CREATE OR REPLACE FUNCTION catalog.schema.get_avg_price(
  neighborhood STRING COMMENT "Neighborhood name, e.g. 'Mission'"
)
RETURNS DOUBLE
LANGUAGE SQL
DETERMINISTIC
COMMENT 'Returns average property price for a SF neighborhood. Use to compare prices.'
RETURN SELECT AVG(price) FROM properties WHERE neighborhood = get_avg_price.neighborhood;
```
```python
from unitycatalog.ai.core.databricks import DatabricksFunctionClient
client = DatabricksFunctionClient(execution_mode="serverless")

def estimate_fare(distance_miles: float, time_minutes: float) -> float:
    """Calculates estimated taxi fare.
    Args:
        distance_miles (float): trip distance in miles, non-negative.
        time_minutes (float): trip duration in minutes, non-negative.
    Returns:
        float: fare in USD = $3 base + $2.50/mile + $0.50/min.
    """
    import math  # import INSIDE the function body — required
    return 3.00 + distance_miles * 2.50 + time_minutes * 0.50

info = client.create_python_function(func=estimate_fare, catalog="cat", schema="sch", replace=True)
```
**Python function rules (hard requirements)**: explicit type hints on every arg + return — **no `*args`/`**kwargs`**; **import external libraries inside the function body**, not at module level (the function is serialized for remote/serverless execution — same root cause as the `@scorer` import rule in §3.5). Allowed hints are primitives and their containers (`int`, `str`, `float`, `bool`, `list`, `dict`, `Optional[T]`) — **Pydantic `BaseModel` types are not supported**.

**Python UC functions can also be declared in pure SQL**, which is how you attach custom dependencies:
```sql
CREATE OR REPLACE FUNCTION cat.sch.my_tool(x STRING)
RETURNS STRING
LANGUAGE PYTHON
ENVIRONMENT (dependencies = '["requests==2.32.3"]', environment_version = 'None')
AS $$
  import requests
  return requests.get(x).text[:200]
$$;
```
The **`ENVIRONMENT` clause** is the mechanism for third-party libraries in a Python UC function.

### Executing a UC function tool directly
```python
result = client.execute_function(function_name="cat.sch.estimate_fare",
                                 parameters={"distance_miles": 5.5, "time_minutes": 15.0})
result.value      # the return value (string form)
result.error      # populated instead of .value if execution failed — always check this
```
**Tool-name translation gotcha**: when UC functions are exposed to an LLM as tools, the dots in the 3-level name become **double underscores** (`cat.sch.fn` → `cat__sch__fn`) because the tool-calling schema disallows dots. Agent code must translate back before executing:
```python
udf_name = tool_name.replace("__", ".")
```
Hand-rolled agent code also strips an unsupported key from the generated spec before sending it to the model — `tool_spec["function"].pop("strict", None)` — and stores each tool in a small `ToolInfo` model holding `name` (str), `spec` (dict) and `exec_fn` (Callable).

### `@function_tool` — the no-UC-registration shortcut
```python
from agents import function_tool

@function_tool
def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    return f"The weather in {city} is sunny, 72F."
```
The decorator builds the tool schema the LLM sees from the function's **name, type hints and docstring** — so clear docstrings and typed parameters directly improve tool-selection accuracy. Use it for quick prototyping or agent-specific logic; use UC functions when you need governance, discoverability and sharing.

## 2.4 Building an agent with LangChain
```python
from databricks_langchain import ChatDatabricks, UCFunctionToolkit
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain.prompts import ChatPromptTemplate
import mlflow
mlflow.langchain.autolog()

llm = ChatDatabricks(endpoint="databricks-gpt-5-1", temperature=0.1, max_tokens=250)
toolkit = UCFunctionToolkit(function_names=["cat.sch.avg_price", "cat.sch.get_details"])
tools = toolkit.tools
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Use tools to answer questions."),
    ("placeholder", "{chat_history}"), ("human", "{input}"), ("placeholder", "{agent_scratchpad}"),
])
agent = create_tool_calling_agent(llm, tools, prompt)
executor = AgentExecutor(agent=agent, tools=tools, verbose=True, max_iterations=10)
response = executor.invoke({"input": "Compare avg price for Mission with listing 958."})
print(response["output"])
```
**Reasoning loop**: Observation → Thought → Action (tool call) → Observation (tool result) → Thought → Final Answer. MLflow traces capture every step.

Newer `create_agent` + `VectorSearchRetrieverTool` (LangGraph-based, supports a `checkpointer` for multi-turn memory keyed by `thread_id`):
```python
from databricks_langchain import VectorSearchRetrieverTool
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

retriever_tool = VectorSearchRetrieverTool(name="kb_search", index_name=f"{catalog}.{schema}.docs_index",
                                           description="Search knowledge base", num_results=5)
agent = create_agent(model=llm, tools=[retriever_tool],
                     system_prompt="Answer using only provided context.",
                     checkpointer=InMemorySaver())
resp = agent.invoke({"messages": [{"role": "user", "content": "What is Orion?"}]},
                    config={"configurable": {"thread_id": "conv-1"}})
```

## 2.5 OpenAI SDK integration (function-calling, not the Agents SDK)
```python
from databricks.sdk import WorkspaceClient
from databricks_openai import UCFunctionToolkit

client = WorkspaceClient().serving_endpoints.get_open_ai_client()
toolkit = UCFunctionToolkit(function_names=["cat.sch.func1"])
response = client.chat.completions.create(
    model="databricks-gpt-5-1",
    messages=[{"role": "user", "content": "..."}],
    tools=[t.spec for t in toolkit.tools], tool_choice="auto")
```

## 2.6 Other supported frameworks
Databricks supports multiple agent authoring frameworks; the key requirement for deployment is **wrapping the agent in the MLflow `ResponsesAgent` interface** (§2.8).
| Framework | Strength |
|---|---|
| **OpenAI Agents SDK** | The course default — handoffs, MCP servers, `@function_tool` |
| **LangChain** | Composable chains and tools |
| **LangGraph** | Stateful graph workflows |
| **DSPy** | Programmatic prompt optimization: **Signatures** describe desired input/output behaviour in natural language, **Modules** implement transformations, a **Compiler** optimizes the pipeline automatically, and a **Program** chains modules together — systematic optimization rather than trial-and-error prompting |

### Why agents need deeper observability than traditional ML
| | Traditional ML | Agents |
|---|---|---|
| Shape | Single call | Multi-step, often looped |
| Visible by default | Input, output | Input, final output only |
| Unit of observation | The call | **A span per step** |
| Needed to debug | Features, latency | Per-step I/O, tokens, tool args |
| MLflow captures | Request / response | **Full trace (tree of spans)** |

MLflow spans follow the **OpenTelemetry** trace specification, so extra information (such as token counts) is stored as **key-value attributes on the span** rather than as new top-level fields. Spans form a hierarchy: a **root span** for the overall request with nested **child spans** per sub-step, mirroring the application's execution plan.

## 2.7 MLflow Tracing for agents — deeper detail
**The five span types you must know**: `AGENT` (autonomous agent operations — usually the root span), `CHAT_MODEL` (LLM interactions), `TOOL` (function calls), `RETRIEVER` (AI Search fetches), `CHAIN` (operation sequences).
**Full span type list** (recognize them): `LLM`, `EMBEDDING`, `GUARDRAIL`, `PARSER`, `RERANKER`, `EVALUATOR`, `MEMORY`, `TASK`, `WORKFLOW`, `UNKNOWN` — or a custom string.
A typical agent trace = root `AGENT` span → child `CHAT_MODEL` / `TOOL` / `RETRIEVER` spans, showing which tools were called, in what order, and how long each took.

```python
import mlflow
from mlflow.entities import SpanType

mlflow.langchain.autolog()   # or mlflow.openai.autolog()

@mlflow.trace(name="Validate Input", span_type=SpanType.TOOL, attributes={"version": "1.0"})
def validate_input(question: str, min_length: int = 5) -> dict:
    if len(question) < min_length:
        return {"valid": False, "error": "too short"}
    return {"valid": True, "cleaned": question.strip()}

# Manual spans for custom logic:
with mlflow.start_span(name="preprocessing") as span:
    cleaned = preprocess(raw)
    span.set_inputs({"raw": raw})
    span.set_outputs({"cleaned": cleaned})

with mlflow.start_active_span("Process Data") as span:
    span.set_attribute("data_size", 42)
```
Each span captures **inputs/outputs, timing (start/end/duration), span type, and status**. If the agent calls a supported library (OpenAI SDK, LangChain), child spans are generated **automatically** — you only add manual spans for your own logic.

⚠ **Some judges cannot run without traces.** `RetrievalGroundedness`, `RetrievalRelevance` and `RetrievalSufficiency` all analyse **`RETRIEVER` spans** — they assess *what was retrieved*, not just the final response. No traces → these evaluations are impossible.

**Tags vs. metadata** — this distinction is exam-relevant:
- **Tags** — mutable, can be set/updated any time during the trace lifecycle via `mlflow.update_current_trace(tags={...})`. Use for env/version labels and filtering.
- **Metadata** — immutable, set once at trace creation. Use for static config, session id, and user id.

⚠ **Multi-turn session grouping requires *metadata*, not tags**: to group traces into one conversation for session-level judges (§3.4), the session id must land in the `mlflow.trace.session` **metadata** key — **setting it as a tag will not be picked up** by multi-turn scorers such as `UserFrustration` or `ConversationCompleteness`.
```python
# Preferred: dedicated parameters (they write into metadata for you)
mlflow.update_current_trace(session_id=session_id, user=user_id)

# Equivalent, explicit form:
mlflow.update_current_trace(metadata={"mlflow.trace.session": session_id,
                                      "mlflow.trace.user": user_id})

mlflow.search_traces(filter_string=f"metadata.`mlflow.trace.session` = '{session_id}'")
```

### Storing traces in Unity Catalog (`trace_location`)
By default traces live in the MLflow control plane. For production you can store the **OpenTelemetry traces directly in UC Delta tables** — long-term retention, SQL-queryable, governed by table permissions instead of experiment ACLs.
```python
from mlflow.entities.trace_location import UnityCatalog

experiment = mlflow.set_experiment(
    experiment_name="my-agent-experiment",
    trace_location=UnityCatalog(catalog_name="my_catalog",
                                schema_name="my_schema",
                                table_prefix="agent_traces"),
)
```
This auto-creates **four Delta tables** under the prefix — **not** one combined table, and not `_input`/`_output`/`_errors`:
| Table | Contents |
|---|---|
| `<prefix>_otel_spans` | Individual span records — the primary trace data |
| `<prefix>_otel_annotations` | Assessments and feedback attached to traces |
| `<prefix>_otel_logs` | Log entries associated with trace execution |
| `<prefix>_otel_metrics` | Metric values computed by scorers |

- **Governance**: users need `USE_CATALOG` + `USE_SCHEMA` + **explicit `MODIFY` and `SELECT` on each table**. `ALL_PRIVILEGES` is **not** sufficient (§3.12).
- **Binding is permanent** — once an experiment is bound to a UC trace location you cannot reassign it. Multiple experiments *can* share one location, and anyone with table access sees all traces there regardless of experiment.
- **Ingestion limits**: 200 traces/second per workspace, 100 MB/second per table.

**Experiment setup (non-UC)**:
```python
mlflow.set_experiment(f"/Workspace/Users/{username}/my_agents")
mlflow.create_experiment(name=experiment_name, artifact_location="dbfs:/Volumes/cat/sch/agent_vol")
```
In a deployed app the experiment name is injected as an env var: `mlflow.set_experiment(os.environ["MLFLOW_EXPERIMENT_NAME"])`.

> **Tracing is the prerequisite for everything downstream.** Scorers evaluate **traces**, not raw requests — if the agent isn't producing traces, there is nothing to score, archive, or backfill.

## 2.8 ResponsesAgent — the production wrapper interface
Production-ready interface matching the OpenAI Responses schema; supports streaming and non-streaming. **Wrapping your agent in `ResponsesAgent` is the requirement for deploying agents built in any framework** (OpenAI Agents SDK, LangChain, LangGraph, DSPy).
```python
from mlflow.pyfunc import ResponsesAgent
from mlflow.types.responses import ResponsesAgentRequest, ResponsesAgentResponse
from uuid import uuid4

class MyWrappedAgent(ResponsesAgent):
    def __init__(self, agent): self.agent = agent
    def predict(self, request: ResponsesAgentRequest) -> ResponsesAgentResponse:
        messages = [i.model_dump() for i in request.input]
        result = str(self.agent.invoke(messages))
        item = self.create_text_output_item(text=result, id=str(uuid4()))
        return ResponsesAgentResponse(output=[item])
    def predict_stream(self, request): ...  # yields ResponsesAgentStreamEvent
```
Helper methods on `ResponsesAgent`: `create_text_output_item(text, id)`, `create_function_call_item(name, arguments, id)`, `create_function_call_output_item(call_id, output)`, and `prep_msgs_for_cc_llm(messages)` (converts Responses-schema messages into the chat-completions format an LLM client expects).

**Request/response schema**:
- Input: `request.input` = `[{"role": "user", "content": "..."}]` — each item is a Pydantic object, so `[i.model_dump() for i in request.input]` to get plain dicts.
- Output: `ResponsesAgentResponse(output=[{"type": "message", "id": str(uuid4()), "role": "assistant", "content": [...]}, ...])`.

**Streaming (`predict_stream`)** yields `ResponsesAgentStreamEvent` objects: incremental `response.output_text.delta` events that all share the **same `item_id`**, followed by a terminal `response.output_item.done` event carrying the complete item. Both `predict` and `predict_stream` should be implemented for a fully functional production agent.
**Streaming errors** surface in the response under `databricks_output.error` with `error_code` and `message` fields — not as a raised Python exception on the client.

Finish `agent.py` with `mlflow.models.set_model(AGENT)` so MLflow knows which object to serve (see §1.15).

## 2.9 Logging, registering, and deploying an agent (full lifecycle)
```python
import mlflow
from mlflow.models.resources import DatabricksFunction, DatabricksTable, DatabricksServingEndpoint

with mlflow.start_run():
    mlflow.set_tags({"framework": "openai", "stage": "dev"})
    logged = mlflow.pyfunc.log_model(
        name="my_agent",
        python_model="agent.py",
        artifacts={"config": "config.json"},
        input_example={"input": [{"role": "user", "content": "example"}]},
        pip_requirements=["databricks-openai", "backoff"],
        resources=[DatabricksFunction(function_name="cat.sch.func1"),
                   DatabricksTable(table_name="cat.sch.tbl"),
                   DatabricksServingEndpoint(endpoint_name="databricks-gpt-5-1")],
    )

mlflow.set_registry_uri("databricks-uc")
registered = mlflow.register_model(model_uri=logged.model_uri, name="cat.sch.my_agent")
mlflow.MlflowClient().set_registered_model_alias(
    name="cat.sch.my_agent", alias="Champion", version=registered.version)
```
Declaring `resources=[...]` is what lets the deployed endpoint obtain **automatic credential pass-through** to those UC objects — omit them and the agent fails at runtime with permission errors.

**Governance rationale**: developers must never push straight to production — the UC Model Registry (3-level namespace, aliases, EXECUTE/SELECT grants, full lineage) is the enforced gate.

**7-step agent lifecycle** (as taught): (1) prepare data & create UC-governed tools, (2) rapid prototype in AI Playground, (3) evaluate & collect feedback (LLM judges + expert review), (4) label data/feedback into benchmarks, (5) iterate — balance accuracy/cost/latency, (6) deploy: register to UC → Model Serving, (7) monitor quality/performance continuously in production.
*(For the four-stage **production** lifecycle — Deployment → Observability → Evaluation → Monitoring — see §4.10.)*

## 2.10 Agent Bricks — declarative agent framework
**Positioning**: Agent Bricks builds agents that *know your data* — schemas, business definitions, custom semantics — rather than generic assistants. It is **open and multi-AI** (OpenAI, Anthropic, Google, open-source models; you can switch models instantly without re-architecting) and provides **unified governance** across data, models, tools and MCP servers.
**Declarative vs. code-first**: declarative = describe *what* the agent should do (point at data, describe persona, deploy); code-first = you write and own all the retrieval/tool logic and its ongoing optimization.
**Core mechanism — ALHF (Agent Learning from Human Feedback), a 4-step loop**: Deploy baseline → Review App collects SME feedback (thumbs up/down, corrections, comments) → system synthesizes feedback into evaluation benchmarks & proposed improvements → optimize prompt/config automatically → loop.
**Underlying platform for every brick**: Unity Catalog (governance), Mosaic AI Model Serving (compute), MLflow Tracing (observability), Agent Evaluation (quality).

**Where to build**: left menu → **Agents** → **Create Agent**. Alternatively prototype in the **AI Playground**, then use **Get code → Export to Databricks Apps** to generate an OpenAI-SDK project you can deploy (§4.10).

### The Agent Bricks types
| Type | Category | Does |
|---|---|---|
| **Knowledge Assistant (KA)** | Interactive | RAG Q&A over documents; auto parse/chunk/embed/cite (§1.16) |
| **Supervisor Agent** (Multi-Agent Supervisor) | Interactive | Coordinates **Genie spaces, Knowledge Assistants, UC functions and external MCP servers** — routes a request to the right specialist (the "Supervisor" from §0.9) |
| **Classification** | Automated | Assigns records/documents to defined categories at scale |
| **Information Extraction (IE)** | Automated | Unstructured docs → structured JSON fields (invoices, contracts) |

⚠ **Currency check**: some workspaces still expose the **legacy Custom LLM** and **legacy Information Extraction** flows. Treat **Custom LLM as legacy** (domain/style-tuned LLM for brand voice or compliance-formatted output) and **Classification** as a current brick type.
**Automated** bricks (Classification, IE) = high-scale batch, minimal human intervention, cost/performance optimized. **Interactive** bricks (KA, Supervisor, Genie) = human-in-the-loop, conversational, real-time.

### Agent Bricks development lifecycle (3 steps)
1. **Specify your problem** — align stakeholders on the task; choose the brick type; identify the Unity Catalog resources: **data** (Delta tables, UC Volumes, AI Search indexes) and **tools/subagents** (Genie spaces, UC functions, external MCP servers, existing agent endpoints). Define success criteria — **groundedness, accuracy, coverage, latency, cost**. All in natural language, no code.
2. **Configure & evaluate on enterprise data** — configure in the UI per brick type; test in the AI Playground or the agent's build page; **turn on MLflow Tracing**; run MLflow GenAI evaluation with LLM judges and custom scorers; compare variants on quality, latency and cost. The platform auto-creates benchmarks, optimizes prompts, does selective fine-tuning and tool selection, builds custom LLM judges and applies reward-model filtering/RLHF.
3. **Continuous improvement** — an endpoint is created automatically; MLflow 3 production monitoring samples production traces; SMEs contribute through the built-in **Examples** and **Guidelines** flows plus Review Apps and MLflow feedback annotations; re-run evaluations to confirm improvement.

**Built-in evaluation**: Agent Bricks auto-integrates MLflow (tracking requests, responses, and inter-agent communication), auto-creates task-specific benchmarks, runs LLM-judge evaluation, folds in human feedback, and supports comparative analysis across versions — you don't wire up `mlflow.genai.evaluate()` yourself.

**Supervisor setup**: Agents → Create Agent → Supervisor Agent → **Add UC Functions / Genie spaces / KAs / MCP servers** as tools → configure name/description/instructions → Deploy. Test in AI Playground ("Review Capabilities" → Authorize → ask a query spanning multiple tools → observe routing). Feedback loop: create a Labeling Session with example queries → add expert **Guidelines** per question → re-test in the Build tab to validate improvement.
⚠ **Supervisor access control**: because the supervisor calls UC Function tools under the hood, **the same UC permission model still applies per-user** — it must never surface data a given user isn't individually granted access to, even if the supervisor itself has broad access.

### Genie — natural-language analytics as an agent capability
- A **Genie Space** is a curated natural-language interface to governed Databricks data, where you define the data context and instructions Genie uses (see §2.2).
- A **Genie agent** is a Genie Space acting as a specialized worker inside a larger system.
- **Two integration paths**: (1) **Genie via managed MCP** — an external agent connects through a pre-configured MCP endpoint and calls Genie as a tool; (2) **Genie as a subagent** in a multi-agent system, routed to by a Supervisor Agent.

## 2.11 Model Context Protocol (MCP)
MCP is an **open, standardized protocol for connecting AI applications to external systems** — "a USB-C port for AI". Without it, every tool integration needs bespoke API wrappers, auth handling and response parsing; with it, any compliant client and server interoperate.

**Three participants (client–server architecture):**
| Participant | Role |
|---|---|
| **MCP Host** | The AI application (e.g. a Databricks App) that manages one or more MCP clients |
| **MCP Client** | A component inside the host holding a dedicated connection to **one** MCP server |
| **MCP Server** | A program exposing tools, resources or prompts to connected clients |

A single host can connect to **multiple servers simultaneously**, each via its own client instance.

**Three primitives an MCP server can expose**: **Tools** (executable functions — the common case for agents), **Resources** (read-only data sources), **Prompts** (reusable interaction templates).

**MCP server categories on Databricks:**
| Category | Backing | Example |
|---|---|---|
| **Managed** | Pre-configured, backed by Databricks features — governed automatically | **Unity Catalog functions, Genie spaces, AI Search indexes, Databricks SQL** |
| **External** | Managed connections to MCP servers hosted outside Databricks | Third-party APIs, SaaS integrations |
| **Custom** | An MCP server you write and host, e.g. as a Databricks App | Domain-specific tools, internal services |
| **Local** | Bundled inside the agent's own container | Dev/offline tools |

The common starting point is a **Managed MCP server backed by UC functions**: write a Python function → register in UC → the managed server exposes it as a tool automatically. No custom agent code needed; MCP handles discovery, invocation and response formatting.

**Managed MCP servers stay UC-governed**: even though access goes *through* MCP, **Unity Catalog permissions are still enforced** — MCP does not bypass or replace UC governance. A deployed app's **service principal must have `EXECUTE`** on the UC functions it reaches through MCP.

**Connection stack**: Agent → OpenAI Agents SDK → `McpServer` (`databricks_openai.agents`) → **MCP Protocol over Streamable HTTP** → Unity Catalog (functions, AI Search).

**URL pattern for a managed (UC-function-backed) MCP server:**
```
/api/2.0/mcp/functions/<catalog>/<schema>
```
(not `/sql/`, not `/genie/`, not `/models/` — those are different endpoint families entirely.)

**Connection lifecycle — four phases:**
1. **Initialization** — client sends `initialize` (protocol version + capabilities), server replies with its capabilities, client sends an `initialized` notification to confirm readiness.
2. **Tool discovery** — client sends **`tools/list`**; server returns every tool with name, description and input schema. The agent uses those descriptions to decide when to call each tool.
3. **Tool invocation** — client sends **`tools/call`** with the tool name and arguments; server executes and returns the result.
4. **Shutdown** — either side closes the connection gracefully.

```python
from databricks.sdk import WorkspaceClient
from databricks_openai.agents import McpServer
from agents import Agent, Runner

# McpServer.from_uc_function MUST be used as an async context manager
async with McpServer.from_uc_function(
    catalog="my_catalog", schema="my_schema",
    workspace_client=WorkspaceClient(), name="uc-functions",
) as uc_server:
    agent = Agent(name="my_agent", model="databricks-claude-sonnet-4-5",
                  mcp_servers=[uc_server])
    result = await Runner.run(agent, "Look up info for Acme Corp")

# Lower-level discovery client:
from databricks_mcp import DatabricksMCPClient
mcp_client = DatabricksMCPClient(server_url=mcp_url, workspace_client=WorkspaceClient())
tools = mcp_client.list_tools()
```
In a deployed app, MCP servers are usually initialized **once at startup** (FastAPI `lifespan`) and shared across requests; the catalog/schema come from injected env vars (§4.10).

⚠ **Runtime tool discovery**: MCP tools are discovered at **runtime** via `tools/list`, not hardcoded at build time. Register a **new** UC function on an already-running MCP server and the deployed agent picks it up **with no redeployment**. When MLflow Tracing is active, every MCP tool call is captured as a `TOOL` span inside the trace.

## 2.12 Multi-agent orchestration & the OpenAI Agents SDK
Orchestration patterns split into two categories by **who decides the next step**:

| Category | Pattern | How it works |
|---|---|---|
| **LLM-driven** | **Handoffs** | A supervisor transfers the whole conversation to a specialist worker, which then owns the response |
| **LLM-driven** | **Agents-as-tools** | A supervisor calls other agents like tools and keeps control, synthesizing their outputs |
| **Code-driven** | **Sequential chaining** | An ordered pipeline where each step consumes the previous step's output |
| **Code-driven** | **Parallel execution** | Several independent agents run concurrently on the same input to cut latency |
| **Code-driven** | **Feedback loop** | A generator agent's draft is repeatedly revised by an evaluator agent until approval or a max-iteration cap |

```python
from agents import Agent, Runner, set_trace_processors   # OpenAI Agents SDK
import mlflow

set_trace_processors([])          # disable the SDK's own tracing so MLflow owns it
mlflow.openai.autolog()

billing_agent = Agent(name="Billing", instructions="Handle billing questions.")
tech_agent = Agent(name="Tech Support", instructions="Handle technical issues.")
supervisor = Agent(
    name="Supervisor",
    instructions="Route the user to the right specialist.",
    handoffs=[billing_agent, tech_agent],   # SDK auto-generates transfer_to_{agent.name} tools
)

result = await Runner.run(supervisor, "My invoice looks wrong")
print(result.last_agent.name)     # e.g. "Billing" — the WORKER produces the final answer
print(result.final_output)        # final answer text only

# Chaining runs while preserving full conversation history:
result2 = await Runner.run(supervisor,
    result.to_input_list() + [{"role": "user", "content": "and one more thing..."}])
```
**Key exam points:**
- `handoffs=[...]` on `Agent(...)` makes the SDK **auto-generate `transfer_to_{agent.name}` tools** — you never hand-write routing tools.
- After a handoff the **worker agent** produces the final response — `result.last_agent.name` names it, and it will *not* be the supervisor.
- `result.to_input_list()` carries the **full conversation history** forward; `result.final_output` is only the final text and loses that history.
- Trace with `mlflow.openai.autolog()` for this SDK (vs. `mlflow.langchain.autolog()` for LangChain/LangGraph).

## 2.13 Unity AI Gateway
> **Beta** — the unified AI Gateway (traffic splitting, coding-agent governance, MCP server support) is in Beta; account admins enable it from the **Previews** page.

The centralized AI **governance layer** that sits between applications and serving endpoints. Requests from a deployed agent pass through it automatically — **no agent code changes required**. It routes to **External Models**, **Databricks-hosted models** (pay-per-token or provisioned throughput) and **MCP servers**.

| Capability | Description |
|---|---|
| **Guardrails** | Enforce safety/content policies on requests *and* responses at the gateway |
| **Rate limiting** | Consumption limits **per endpoint and per identity** (user or group) in QPM/TPM, to manage capacity and control cost |
| **Usage tracking** | Cost analysis by endpoint, model, principal and tags via billable-usage system tables. In the new Gateway (Beta) usage tracking is **on by default** once the Gateway is enabled and the endpoint created |
| **Traffic splitting** | Distribute requests across model backends for A/B testing or gradual rollout |
| **Inference tables** | Log requests/responses to UC Delta tables for audit and analysis (§4.8) |
| **Fallbacks** | Automatic failover on **429 / 5xx**: primary model → second model → third model, in the configured order. All attempts and outcomes are logged to the inference/usage-tracking tables |
| **MCP governance** | **On-behalf-of execution**: agents call MCP tools with the **requesting user's exact permissions**, not a shared service account — so UC grants stay per-user end to end |

Scope is broader than agents: one governance layer across **LLM endpoints, MCP servers, and coding agents** (Cursor, Claude Code, Codex CLI) — a single place for platform teams to manage AI access.
Enabled per-endpoint via **Configure AI Gateway** in the Model Serving endpoint UI (also where inference-table logging is switched on).

---
# 3. Agent Evaluation on Databricks

## 3.1 Why traditional testing fails for agents
Traditional testing assumes **same input → same output**, verified by **exact string/number match**, over **explicitly programmed** behaviour with **anticipated** edge cases. Agents break every one of those assumptions:

| Agents break it via | Because |
|---|---|
| **Non-determinism** | Temperature/sampling means `assert output == "expected"` fails even for good answers |
| **Emergent behavior** | Agents choose tools and reasoning paths autonomously — not explicitly programmed |
| **Context dependency** | Responses depend on retrieval, conversation history and external data |
| **Qualitative assessment** | Success needs judgment (helpfulness, tone, completeness), not string equality |

*Example*: "What's the weather in San Francisco?" → "It's currently 65°F and sunny in San Francisco." / "San Francisco weather: 65 degrees, clear skies." / "The temperature in SF is 65°F with no clouds." All three are correct; none match exactly; `assert output == expected` fails all three.

**Four dimensions that make agent evaluation multi-dimensional:**
1. **Multi-step reasoning** — assess the quality of *intermediate* steps, not just the final answer.
2. **Tool-calling accuracy** — right tool? right parameters? correctly interpreted result?
3. **Retrieval quality** — did retrieval surface relevant documents, and did the agent synthesize across them correctly?
4. **Safety & alignment** — avoid harmful output, respect boundaries, decline inappropriate requests.
Plus **real-world variability**: production agents meet diverse queries, unexpected phrasings and edge cases you cannot fully anticipate.

**Evaluation is a continuous cycle, not a one-time gate:**
Development (rapid iteration, frequent evaluation, catch regressions) → Pre-deployment (comprehensive validation against a quality bar) → Production (continuous monitoring for degradation and emerging failure patterns) → **Dataset evolution** (feed real failures back in) → repeat.

**Operational setup requirements** (make results comparable and auditable):
| Requirement | What to do |
|---|---|
| **MLflow Experiments & Runs** | Stable experiment names; tag runs with agent version, dataset version and parameters; compare metrics in the UI |
| **Unity Catalog integration** | Govern datasets and traces with access control, versioning and lineage; register agents for end-to-end traceability |
| **Production feedback loop** | Enable Unity AI Gateway inference tables to log requests, responses and traces for monitoring and mining new evaluation examples |

**MLflow's five capabilities for agent evaluation**: (1) **Tracing** — capture every step; (2) **Scorers & Judges** — automated quality assessment; (3) **`mlflow.genai.evaluate()`** — orchestrate evaluation runs; (4) **Evaluation experiments** — compare and track results over time; (5) **Production monitoring** — continuous evaluation of live traffic.

## 3.2 The three components of `mlflow.genai.evaluate()`
1. **Evaluation dataset** — minimum shape `{"inputs": {"input": "..."}}`; a record may also carry `outputs`, `expectations`, `source` and `tags`. Stored as JSON files, Pandas DataFrames, or (preferably) **UC Delta tables**, versioned alongside the agent.
2. **Scorers/judges** — built-in judges, guideline judges, custom LLM judges (`make_judge`), code-based scorers (`@scorer`), third-party scorers, or multi-turn session judges.
3. **Predict function** — `predict_fn=lambda input: agent.predict({"input": input})`; omit entirely if evaluating pre-generated "answer sheet" outputs.

**Evaluation dataset record fields:**
| Field | Required | Description |
|---|:-:|---|
| `inputs` | ✓ | The request sent to the agent (dict of argument name → value) |
| `outputs` | ✗ | Pre-generated response — enables "answer sheet" mode |
| `expectations` | ✗ | Ground truth: `expected_facts`, `expected_response`, per-row `guidelines` |
| `source` | ✗ | Provenance of the record (human, synthetic, production trace) |
| `tags` | ✗ | Per-record metadata for filtering and slicing |

```python
import mlflow
from mlflow.genai.scorers import Correctness

results = mlflow.genai.evaluate(
    data=eval_dataset,
    predict_fn=lambda input: agent.predict({"input": input}),
    scorers=[Correctness(model="databricks:/foundation-model-endpoint")],
    model_id="m-abc123...",     # optional: link this run to a LoggedModel
)
print(results.run_id, results.metrics)
```

**`mlflow.genai.evaluate()` key parameters:**
| Parameter | Required | Description |
|---|:-:|---|
| `data` | ✓ | The evaluation dataset |
| `scorers` | ✓ | List of scorers/judges to apply |
| `predict_fn` | ✗ | Agent function to invoke; omit for answer-sheet mode |
| `model_id` | ✗ | Links results to a specific LoggedModel for version tracking |

⚠ **`predict_fn` requirements (high-value exam detail)** — the function must:
- accept the **keys of the `inputs` dict as keyword arguments**,
- return a **JSON-serializable dictionary**,
- be **instrumented with MLflow Tracing**, and
- emit **exactly one trace per call**.
If it isn't already traced, MLflow **automatically applies `@mlflow.trace`** for you.

**What `EvaluationResult` gives you:**
| Attribute | Contents |
|---|---|
| `run_id` | MLflow run identifier for the evaluation |
| `metrics` | Aggregated metrics dictionary across all examples |
| `artifacts` | Artifacts produced by the run |
| `tables` | Per-example result DataFrames |

**Per-example results in MLflow 3** come from the traces, not a `.result_df`:
```python
eval_traces = mlflow.search_traces(run_id=results.run_id, return_type="list")
for trace in eval_traces:
    for assessment in trace.info.assessments:
        print(assessment.name, assessment.value)
```

**The five-step execution workflow inside `evaluate()`**: Data Loading (validate structure) → Output Generation (call `predict_fn`, or use supplied outputs) → Trace Creation (one trace per example) → Scorer Execution (each scorer runs against inputs/outputs/traces) → Aggregate & Log (metrics written to the MLflow run).
**Each evaluation run contains**: **Traces** (one per input), **Feedback** (assessments on each trace), **Metrics** (aggregates), **Metadata** (config used).

**Two evaluation modes — know the difference:**
- **Direct evaluation** (`predict_fn` supplied): the agent actually runs during evaluation and generates fresh traces.
- **"Answer sheet" mode** (`predict_fn` omitted, dataset carries an `outputs` field): MLflow scores **pre-generated** outputs and synthesizes traces from the inputs/outputs. Use this to bulk-score existing conversation logs against a new judge without re-running inference.

**Closing the loop with inference tables**: Production Agent → **AI-Gateway-enabled inference tables** → extract failures and edge cases → augment the evaluation dataset → re-evaluate. Benefits: automatic logging (no extra instrumentation), rich metadata (timestamps, latency, tokens), SQL-queryable in Unity Catalog, and directly usable as evaluation datasets.

**Module hierarchy — which import to use:**
- `mlflow.genai.scorers` — the orchestration layer used by `mlflow.genai.evaluate()`. Contains the built-in judge classes (`Correctness`, `Safety`, `Guidelines`, ...), the `@scorer` decorator, `ScorerSamplingConfig`, and the management functions `list_scorers` / `get_scorer` / `delete_scorer`.
- `mlflow.genai.judges` — the narrower LLM-judge layer. This is where **`make_judge`** lives (also re-exported as `from mlflow.genai import make_judge`).

**Scorers vs. traditional ML metrics**: a scorer returns a structured **`Feedback` object** (a value *plus* a natural-language rationale) and assesses qualitative dimensions; a traditional metric (accuracy/F1/RMSE) returns one scalar with no explanation of *why* something failed.

**OpenTelemetry**: MLflow traces are OTel-compatible. Three export modes: MLflow tracking only (default), OpenTelemetry only, or dual export — enabling export to Datadog / New Relic / Grafana / Splunk alongside (or instead of) MLflow.

## 3.3 The judge spectrum — five levels of customization
| Customization | Type | Import / API | Use when |
|---|---|---|---|
| **Minimal** | **Built-in judges** | `mlflow.genai.scorers` | Standard quality dimensions (Correctness, Safety, RetrievalGroundedness) — zero prompt engineering |
| **Moderate** | **Guideline judges** | `Guidelines` / `ExpectationsGuidelines` | Natural-language rules for style, tone, compliance, factuality |
| **Full** | **Custom LLM judges** | `make_judge()` | Scores, categories or booleans from subjective reasoning you define |
| **Full (deterministic)** | **Code-based scorers** | `@scorer` decorator | Exact match, format validation, computed metrics |
| **Full (external)** | **Third-party scorers** | Open-source eval frameworks | Reuse an existing external evaluation library |

**The standard production pattern is always two steps — `.register()` then `.start()`:**
```python
from mlflow.genai.scorers import Safety, ScorerSamplingConfig

safety = Safety().register(name="my_safety_judge")                        # bind to active experiment
safety = safety.start(sampling_config=ScorerSamplingConfig(sample_rate=0.7))  # begin evaluating traces

custom = Safety(model="databricks:/databricks-gpt-oss-20b").register(name="custom_safety_judge")
```
`.register(name=...)` must be **unique within the experiment**. Every built-in judge accepts an optional `model=` override in the form `"databricks:/<endpoint-name>"` (also `"openai:/gpt-4o"` or a LiteLLM-style `"provider/model-name"`).

### Built-in judges — full catalogue
Built-in judges are **research-validated**: developed through research, validated against human expert judgment, and optimized for their specific criterion.

| Category | Judge | Requires | Evaluates |
|---|---|---|---|
| Safety | `Safety` | — | Free of harmful, offensive or unsafe content |
| Response quality | `RelevanceToQuery` | — | Response directly addresses the user's query |
| | `Correctness` | **`expectations`** | Factual correctness vs. ground truth |
| RAG | `RetrievalRelevance` | **traces with `RETRIEVER` spans** | Are retrieved documents relevant to the query |
| | `RetrievalGroundedness` | **traces with `RETRIEVER` spans** | Is the response grounded in retrieved context, not hallucinated |
| | `RetrievalSufficiency` | **`expectations` + `RETRIEVER` spans** | Did retrieved context contain enough info for the ground-truth answer |
| Tool calls | `ToolCallEfficiency` | — | Were tool calls efficient / non-redundant |
| | `ToolCallCorrectness` | Optional ground truth | Were the right tools called with the right args |
| Guidelines | `Guidelines` | rules supplied (not ground truth) | Uniform natural-language criteria, same for every row |
| | `ExpectationsGuidelines` | per-row `expectations.guidelines` | Per-example unique criteria |

⚠ **Retrieval judges are trace-dependent** — they inspect `RETRIEVER` spans. Without tracing enabled you simply cannot run `RetrievalGroundedness`, `RetrievalRelevance` or `RetrievalSufficiency`.

**Typical four-step usage**: (1) Define your agent (traced function) → (2) choose a scorer → (3) build the dataset (inputs + expectations) → (4) call `mlflow.genai.evaluate()`.

### Guideline judges — global vs. per-row
| | **Global — `Guidelines()`** | **Per-row — `ExpectationsGuidelines()`** |
|---|---|---|
| Rules | The **same** rules applied to every row | **Different** rules per example |
| Typical use | Tone, style, formatting, compliance | Scenario-specific validation criteria |
| Source of rules | The `guidelines=[...]` constructor arg | Each row's `expectations.guidelines` |

```python
from mlflow.genai.scorers import Guidelines, ScorerSamplingConfig

tone_guidelines = Guidelines(
    name="professional_tone",
    guidelines=[
        "The response must use professional language",
        "The response must not use slang",
        "The response must address the user respectfully",
    ],
    model="databricks:/foundation-model-endpoint",
)
results = mlflow.genai.evaluate(data=eval_data, scorers=[tone_guidelines])

english_judge = Guidelines(name="english",
                           guidelines=["The response must be in English"]).register(name="is_english")
english_judge = english_judge.start(sampling_config=ScorerSamplingConfig(sample_rate=0.7))
```
Guideline judges return `Feedback` with value **`"yes"` / `"no"` (strings, not booleans)**.

**⚠ The key rule for writing guidelines: refer to inputs as "the request" and outputs as "the response".** The judge extracts request/response data from the trace automatically, so guidelines must be framed in those terms.
Recommended phrasing: **"The response must …"** (required) · **"The response must not …"** (prohibited) · **"The response may optionally …"** (suggested).
Additional tips: be specific and concrete ("The response must cite the source document" > "The response should be credible"); write objectively verifiable rules; focus on observable attributes of the response; leave as little ambiguity as possible; test on multiple examples.
Why guideline judges: **expert-accessible** (no coding), **rapid iteration** (change rules without code changes), **interpretable** (self-documenting), **flexible** (complex context-dependent rules).

**`ExpectationsGuidelines` requires an `outputs` field** in the dataset — supplied directly on the row or carried in a trace. It does **not** require a `predict_fn`, and does **not** need registering in Unity Catalog first.

## 3.4 Multi-turn (conversation) judges
Multi-turn judges evaluate an **entire conversation**, not one request. They operate on **sessions** — groups of traces sharing the same `mlflow.trace.session` metadata value — and the resulting **assessment attaches to the *first* trace in the session**.

| Judge | What it evaluates |
|---|---|
| `ConversationCompleteness` | Whether the agent fully resolved the user's request across all turns |
| `UserFrustration` | Repeated questions, escalation language, confusion signals |
| `ConversationalSafety` | Safety violations that only emerge across multiple turns |
| `KnowledgeRetention` | Whether the agent retains information from earlier in the conversation |
| `ConversationalGuidelines` | Whether responses comply with guidelines throughout the conversation |
| `ConversationalRoleAdherence` | Whether the agent maintains its assigned role throughout |
| `ConversationalToolCallEfficiency` | Whether tool usage across the conversation was efficient |

All seven are **built-in**, need **no external integrations**, and **none require ground truth**.
```python
from mlflow.genai.scorers import ConversationCompleteness, ScorerSamplingConfig

completeness = ConversationCompleteness().register(name="conversation_completeness")
completeness = completeness.start(sampling_config=ScorerSamplingConfig(sample_rate=1.0))
```

**Session configuration (exam-critical):**
- Set via `mlflow.update_current_trace(session_id=..., user=...)` — the dedicated parameter writes into the `mlflow.trace.session` **metadata** key automatically. The explicit `metadata={...}` dict is equivalent.
- **Metadata is immutable** once logged; tags are mutable. Session id set as a **tag will be ignored** by the monitoring service.
- A conversation is treated as complete after an inactivity buffer, **default 5 minutes**, configurable via `MLFLOW_ONLINE_SCORING_DEFAULT_SESSION_COMPLETION_BUFFER_SECONDS`.

```python
traces = mlflow.search_traces(
    filter_string=f"metadata.`mlflow.trace.session` = '{session_id}'",
    return_type="list",   # "list" of Trace objects, or "pandas" for a DataFrame
)
```

**Single-turn vs multi-turn — how the catalogue splits:**
| | Scorers |
|---|---|
| **Single-turn, no ground truth** | `RelevanceToQuery`, `RetrievalRelevance`, `RetrievalGroundedness`, `Safety`, `Guidelines`, `ToolCallEfficiency` |
| **Single-turn, ground truth required** | `Correctness`, `RetrievalSufficiency`, `ExpectationsGuidelines` |
| **Multi-turn (session metadata required)** | `ConversationCompleteness`, `ConversationalGuidelines`, `ConversationalRoleAdherence`, `ConversationalSafety`, `ConversationalToolCallEfficiency`, `KnowledgeRetention`, `UserFrustration` |

**Why multi-turn matters**: a single-turn safety check may pass every individual response, yet a user repeating the same question four times signals a broken experience that only a conversation-level judge can detect.

**How built-in judges are validated**: Databricks establishes a human baseline with expert annotators, then measures judge agreement using **Cohen's Kappa**, accuracy and F1, cross-validated across domains — hence "research-validated" rather than ad-hoc prompts.

## 3.5 Custom judges and custom scorers

### Custom LLM judges — `make_judge()` (declarative)
Use when evaluation needs **subjective reasoning** that is hard to express as deterministic code: domain-specific criteria, nuanced assessment beyond built-in judges, proprietary/regulatory standards, or logic combining multiple signals. You describe *what* to evaluate in natural language; MLflow handles the LLM invocation, response parsing, structured-output enforcement and `Feedback` generation.
```python
from typing import Literal
from mlflow.genai.judges import make_judge          # or: from mlflow.genai import make_judge

formality_judge = make_judge(
    name="formality",
    instructions="""Evaluate whether the response is formal, somewhat formal, or not formal.
Request: {{ inputs }}
Response: {{ outputs }}""",
    model="databricks:/databricks-claude-sonnet-4-5",
    feedback_value_type=Literal["formal", "semi_formal", "not_formal"],
).register(name="formality_judge")

# Trace-based judge — `model` is REQUIRED here:
tool_judge = make_judge(
    name="tool_usage_validator",
    instructions="Did the agent pick the right tool with correct params?\n\nTrace: {{ trace }}",
    feedback_value_type=bool,
    model="databricks:/foundation-model-endpoint",
)

feedback = tool_judge(trace=trace_object)     # direct call for a spot check
print(feedback.value, feedback.rationale)
```
Use `feedback_value_type=Literal["yes", "no"]` for a simple pass/fail judge, or a wider `Literal[...]` for multi-category output.

**Only five template variables are valid**: `{{ inputs }}`, `{{ outputs }}`, `{{ expectations }}`, `{{ trace }}`, `{{ conversation }}` (usable only together with `expectations`, not with `inputs`/`outputs`/`trace`). Custom variables like `{{ question }}` **raise a validation error**; at least one allowed variable must be present.

| Parameter | Required | Description |
|---|---|---|
| `name` | Yes | Judge identifier (becomes the metric name) |
| `instructions` | Yes | Natural-language criteria; must contain ≥1 valid template variable |
| `model` | Conditional | **Required for `{{trace}}`-based judges**; optional otherwise |
| `description` | No | What the judge evaluates |
| `feedback_value_type` | Recommended | Type of `Feedback.value` |
| `inference_params` | No | Model params, e.g. `{"temperature": 0.1, "top_p": ..., "max_tokens": ...}` |

`feedback_value_type` supports `int` (e.g. 1–5 rating), `float` (0.0–1.0), `str`, `bool`, `Literal[...]` (enum choices), `dict[str, <primitive>]`, `list[<primitive>]` — **Pydantic `BaseModel` types are not supported**.

### Code-based scorers — `@scorer` (imperative)
Use when the criterion is **deterministic**: format checks, length validation, keyword matching, schema verification. You write the logic in Python and keep full control of scoring, data access and error handling.

**Accepted keyword arguments — your function may take any combination:**
| Parameter | Type | Description |
|---|---|---|
| `inputs` | `dict[str, Any]` | The app's raw input (argument names → values) |
| `outputs` | `Any` | The app's raw output |
| `expectations` | `dict[str, Any]` | Ground-truth labels (expected facts, guidelines, ...) |
| `trace` | `mlflow.entities.Trace` | The complete trace with all spans and metadata |

```python
from mlflow.genai.scorers import scorer
from mlflow.entities import Feedback

# Primitive return — simple pass/fail, no rationale
@scorer
def mentions_databricks(outputs):
    return "databricks" in str(outputs.get("response", "")).lower()

# Feedback return — value + rationale, fully interpretable
@scorer
def response_length_ok(outputs):
    """Verify response length is appropriate."""
    word_count = len(str(outputs.get("response", "")).split())
    if 20 <= word_count <= 100:
        return Feedback(value="yes",
                        rationale=f"Response length ({word_count} words) is appropriate")
    return Feedback(value="no",
                    rationale=f"Response is too {'short' if word_count < 20 else 'long'}")

@scorer(aggregations=["mean", "min", "max"])
def response_length(outputs):
    return len(str(outputs.get("response", "")))
```
`aggregations=["mean", "min", "max"]` on numeric scorers produces experiment-level summary statistics for trend analysis.

> **Both `@scorer` and `make_judge()` work in offline evaluation (`mlflow.genai.evaluate()`) and in production monitoring (`.register()` + `.start()`, `ScorerScheduleConfig`).**

**Serialization rules — custom scorers run remotely and must be fully self-contained:**
| Rule | Details |
|---|---|
| Imports inline | Every `import` must be **inside** the function body |
| Notebook-defined | Must be defined in a Databricks notebook |
| No class-based scorers | `Scorer` subclasses cannot be serialized for remote execution |
| No external references | Cannot reference variables/modules defined outside the function |
| No import-requiring type hints | Signature hints cannot reference types that need imports |

## 3.6 Feedback, expectations and human review

### The `Feedback` object — four fields
| Field | Contents |
|---|---|
| `value` | `"yes"` / `"no"` / a score / a category |
| `rationale` | Human-readable explanation of the decision |
| `source` | An `AssessmentSource` (type + id) |
| `metadata` | Additional context (dict) |

```python
from mlflow.entities import Feedback

Feedback(value="yes", rationale="Correctly identifies Sacramento as capital.",
         metadata={"confidence": 0.95})
```
**`source` is auto-populated**: `CODE` for `@scorer`, `LLM_JUDGE` for `make_judge()`. Set it manually only when you call your own LLM from inside a `@scorer`.

**Why rationales matter**: **debugging** (why it failed, not just that it failed) · **judge validation** (check the reasoning is sound, not just the verdict) · **pattern identification** (recurring themes reveal systemic issues) · **communication** (explain results to stakeholders).
Practice: read rationales for **all failures** to find patterns; **spot-check passes** so a judge isn't agreeing by luck; extract common rationale phrases to categorize failure types; share representative rationales when discussing results.

⚠ **Two MLflow Assessment types attach to a trace**: **Feedback** assessments evaluate the app's actual outputs/intermediate steps ("was the response good?" — what scorers/judges produce). **Expectation** assessments define the desired/correct outcome (ground truth) the app *should* have produced (what `expected_facts`/`expected_response`/`guidelines` populate). Both can be added programmatically or via the MLflow UI's Assessment panel.

### Who gives feedback — three reviewer personas
| Persona | Access needed | How they review |
|---|---|---|
| **Developer** | Full workspace access | Annotates traces directly in the MLflow UI during development |
| **Domain expert / SME** | Account access but **not** workspace access; `CAN_QUERY` on the serving endpoint + `CAN_EDIT` on the MLflow Experiment | Uses the **Review App** chat UI to test the agent and leave structured feedback |
| **End user** | None (production user) | Thumbs up/down or comments in the embedded app UI, logged onto the trace |

### MLflow Review App (SME feedback collection)
```python
import mlflow
import mlflow.genai.labeling as labeling

mlflow.set_tracking_uri("databricks")
review_app = labeling.get_review_app()
review_app.add_agent(agent_name="my_agent", model_serving_endpoint=endpoint_name)
print(f"{review_app.url}/chat")     # share this URL with SMEs

exp = mlflow.get_experiment_by_name(experiment_name)
df = mlflow.search_traces(locations=[str(exp.experiment_id)], include_spans=True, return_type="pandas")
```

### Deploying an evaluated agent — `agents.deploy()`
```python
from databricks import agents

deployment = agents.deploy(model_name="cat.sch.my_agent",
                           model_version=int(version), scale_to_zero=True)
print(deployment.endpoint_name)
```
This is the Agent-Framework shortcut that provisions the serving endpoint **and** the Review App in one call — as opposed to hand-building an `EndpointCoreConfigInput` (§4.4). `scale_to_zero=True` saves cost but makes the first query after idle slow.

## 3.7 Offline vs. online evaluation
| | Offline (pre-deployment) | Online (production) |
|---|---|---|
| Data | Curated datasets, controlled and reproducible | Real production traffic, continuous |
| Ground truth | Available (expert / synthetic / mined) | Rare or absent |
| Purpose | Quality gate, A/B comparison, hypothesis testing, rapid iteration | Drift detection, scale validation, real-world quality signal |
| Strengths | Rigorous validation before users see the agent; A/B testing configurations; baseline metrics for production | Reflects actual user experience; detects unanticipated issues; supplies data for continuous improvement |
| Limitations | May not represent real user behaviour; static datasets go stale; can't capture scale/diversity issues | Users may hit failures first; harder to attribute root causes; requires production infrastructure |

**Offline workflow (5 steps)**: Curate dataset (use cases + edge cases) → Define expectations (optional ground truth/guidelines) → Run agent (generate responses) → Apply scorers (multiple judges) → Analyze & iterate (compare results).

**Online workflow (5 steps)**: Deploy with tracing (Unity AI Gateway) → Capture traces (into an MLflow experiment) → Automatic scoring (`.register()` + `.start()`) → Alerts & feedback (quality monitoring) → Augment dataset (with production examples).

**Production monitoring capabilities**: **auto scoring** (LLM judges on production traces), **sampling** (configurable sample rates), **alerting** (quality-threshold alerts), **trace inspection** (deep dive into failures).

⚠ **Which scorer types support production monitoring**: built-in judges, `make_judge()` judges, `Guidelines`, and `@scorer`-decorated functions **all** work via `.register()` + `.start()`. **Only direct `Scorer` subclasses and third-party scorers are excluded.**

**When to use offline**: quality gates before users see the agent · A/B testing alternative implementations · hypothesis testing ("does the new prompt reduce hallucinations?") · rapid iteration during development.
**When to use online**: real-world quality signal · detecting issues you never anticipated · mining new evaluation examples.

**Feedback loop (5 steps)**: Offline eval on curated datasets → Deploy with monitoring enabled → Analyze traces for failures and edge cases → Augment the dataset with real examples → Re-evaluate, validate and redeploy.

**Unity Catalog as the governance layer for evaluation**: **agent registration** (dependencies, access controls, aliases) and **dataset storage** (versioned, governed, lineage-tracked) — giving traceability across agents, datasets and traces.

## 3.8 Evaluation dataset best practices
Representativeness (match real query distribution — common, high-impact queries so offline results generalize) + edge-case coverage (ambiguous, out-of-scope, adversarial prompts to surface failure modes early) + diversity (length, complexity, domain, user expertise) + appropriate ground-truth strategy (expected answers/fact sets for objective questions; natural-language guidelines where style, policy or completeness matter) + **storage and versioning in Unity Catalog** (built-in versioning, lineage, sharing and governance).

**Ground-truth sourcing trade-offs**: expert labeling (most accurate, resource-intensive) · synthetic generation (scalable, must be validated) · production mining (most realistic, needs filtering).
**Maintenance cadence**: add new production examples, remove/update stale ones, rebalance categories as usage shifts, and version the dataset alongside the agent version.

**Seven quality dimensions to evaluate against** (use this to decide *which* scorers you need): **Correctness** (factual accuracy), **Relevance** (answers the actual question), **Completeness** (thorough without padding), **Safety** (no harmful content), **Consistency** (similar quality for similar queries), **Efficiency** (appropriate tool/resource use), **User experience** (clear, helpful, well-formatted).

**Workflow best practices**: automate evaluation in CI/CD on code changes · set explicit quality gates/thresholds for deployment · always compare against a baseline · document rationales behind decisions · share results with stakeholders · act on the insights.

## 3.9 Online evaluation — sampling strategy and scorer lifecycle

### Sampling strategy
The higher the risk of missing an issue, the higher the sample rate:
| Tier | Rate | Rationale |
|---|---|---|
| **Critical** (safety, security) | **100%** (`sample_rate=1.0`) | Cost of missing a violation far outweighs evaluation cost |
| **Expensive** (LLM-based judges) | **5–10%** (`0.05`–`0.10`) | Still statistically meaningful quality signal at scale |
| **Custom** (lightweight function scorers) | Variable, can be higher | Cheap to run; combine single-turn and multi-turn at different rates |

**Exam scenario**: high-traffic agent needing zero-miss safety coverage *and* a cost-controlled read from an expensive LLM judge → safety at **100%**, expensive judge at **~5–10%** — not 50/50, not both at 100%, not inverted.

### Scorer lifecycle — the state machine
`Created` (in code only) → `.register()` → `Registered` (known to experiment, not yet evaluating) → `.start()` → `Running` ⇄ `.stop()` / `.start()` ⇄ `Stopped` → `delete_scorer()` → `Deleted`.

| Method | Effect |
|---|---|
| `.register(name=...)` | Binds to the active MLflow experiment; name unique per experiment; appears in the **Scorers** tab but does not evaluate yet |
| `.start(sampling_config=...)` | Activates evaluation of **new** traces at the given sample rate |
| `.update(sampling_config=...)` | Changes sampling on a running scorer |
| `.stop()` | Pauses evaluation; stays registered and restartable; historical assessments preserved |
| `delete_scorer(name=...)` | Permanent removal — **must be stopped first**; historical assessments remain on their traces |

⚠ **Immutability pattern**: every lifecycle method returns a **new instance** — always reassign. `safety.stop()` without reassignment leaves your variable pointing at the old (running) state.
```python
from mlflow.genai.scorers import (Safety, ScorerSamplingConfig,
                                  list_scorers, get_scorer, delete_scorer)

safety = Safety().register(name="safety_v1")
safety = safety.start(sampling_config=ScorerSamplingConfig(sample_rate=1.0))
safety = safety.update(sampling_config=ScorerSamplingConfig(sample_rate=0.5))
safety = safety.stop()
delete_scorer(name="safety_v1")
```

### Scorer management
```python
# Audit everything currently running and at what rate
for s in list_scorers():
    if s.sample_rate > 0:
        print(f"{s.name} active at {s.sample_rate}")

# Fetch a scorer created in the UI or another notebook session
my_scorer = get_scorer(name="professional")          # resolves against the ACTIVE experiment
updated = my_scorer.update(sampling_config=ScorerSamplingConfig(sample_rate=0.8))
print(my_scorer.sample_rate, updated.sample_rate)    # original unchanged — immutable

# Bulk cleanup
for s in list_scorers():
    s = s.stop()
    delete_scorer(name=s.name)
```
- **Maximum 20 scorers per experiment.**
- `get_scorer()` works for **UI-created and API-created** scorers alike, and the returned object supports all lifecycle methods.
- A background **Trace Metrics Computation Job** keeps running until **all** scorers in the experiment are stopped.

## 3.10 `mlflow.search_traces()` filter syntax
Correct filter expressions use dotted trace attributes with SQL-like operators:
```python
mlflow.search_traces(filter_string="trace.status = 'ERROR'")
mlflow.search_traces(filter_string="trace.execution_time_ms > 5000")
mlflow.search_traces(filter_string=f"metadata.`mlflow.trace.session` = '{session_id}'")
mlflow.search_traces(run_id=results.run_id, return_type="list")
```
(NOT `trace.state = failed`, NOT `duration > 5s` — those are not valid filter syntax.)

## 3.11 Metric backfill vs. trace archival — two different capabilities
Do not conflate these — they solve different problems.

### Metric backfill — apply a *new* scorer to *old* traces
Assessments are written back onto the **original historical traces**; the original requests are never re-run.
```python
from databricks.agents.scorers import backfill_scorers, BackfillScorerConfig
from datetime import datetime, timedelta

# Simple: use each scorer's currently registered sample rate
job_id = backfill_scorers(scorers=["safety_check", "response_length"])

# Custom rates + time window
job_id = backfill_scorers(
    experiment_id=YOUR_EXPERIMENT_ID,
    scorers=[BackfillScorerConfig(scorer=safety_judge, sample_rate=0.8),
             BackfillScorerConfig(scorer=response_length, sample_rate=0.9)],
    start_time=datetime.now() - timedelta(days=7),
    end_time=datetime.now(),
)
```
- Import is **`databricks.agents.scorers`** (not `mlflow.genai.scorers`).
- Creates a **`[experiment_id] Metric Backfill Computation Job`** in Jobs & Pipelines and returns a **`job_id`** for tracking.
- Pass a list of **scorer name strings** to reuse their current sample rates, or `BackfillScorerConfig` objects for custom rates.
- **Only one backfill job per experiment at a time** — starting a second raises **`RESOURCE_CONFLICT`**.
- Best practice: test on a narrow time range first, lower the sample rate for expensive judges, verify names with `list_scorers()`.

### Trace archival — stream traces to a Delta table
```python
from mlflow.tracing.archival import (enable_databricks_trace_archival,
                                     disable_databricks_trace_archival)

enable_databricks_trace_archival(
    delta_table_fullname="my_catalog.my_schema.archived_traces",
    experiment_id="YOUR_EXPERIMENT_ID",
)
disable_databricks_trace_archival(experiment_id="YOUR_EXPERIMENT_ID")
```
- Module is **`mlflow.tracing.archival`**; creates a **`[experiment_id] Trace Archive Job`** that continuously streams traces.
- The Delta table is **created automatically** and its schema is managed for you — no manual DDL.
- Writes are **append-only**; existing data is never overwritten.
- Disabling stops the job but **preserves** the table; archival can be re-enabled at any time.
- Can also be switched on from the MLflow Experiment UI (click **"Delta sync: Not enabled"**).
- Purpose: SQL queries, Lakeview dashboards, joining traces with business data.

> **Backfill** = "I just wrote a new judge, score my historical traces too." **Archival** = "I need to query/report on raw trace history in SQL." Note this is distinct again from **`trace_location=UnityCatalog(...)`** (§2.7), which stores traces in UC tables *natively* from the start.

## 3.12 UC governance on trace tables
`ALL_PRIVILEGES` **alone is not sufficient** for an analyst to read UC-backed agent trace tables. Required grants: `USE_CATALOG`, `USE_SCHEMA`, **plus explicit `SELECT`** (and `MODIFY` if they need to write) **on each table** — catalog/schema-level `USE` grants don't cascade table read access, and `ALL_PRIVILEGES` at a higher scope doesn't substitute for the explicit table-level grants.

## 3.13 Deploying the evaluated agent
Two deployment patterns exist: **registered-model → Model Serving endpoint** (§4.4) and **application-code → Databricks App** (§4.10). Full detail of the app pattern — DABs, the two-step CLI deploy, and resource injection — is in **§4.10**.

---
# 4. Module 4 — Deployment & Monitoring (W4 + W5)

## 4.1 Model Registry: loading, aliases, versions
```python
mlflow.set_registry_uri("databricks-uc")   # 3-level namespace: catalog.schema.model_name
model = mlflow.pyfunc.load_model(model_uri=f"models:/{model_name}@champion")   # by alias
model = mlflow.pyfunc.load_model(model_uri=f"models:/{model_name}/{version}")  # by version
client = mlflow.MlflowClient()
client.set_registered_model_alias(name=model_name, alias="champion", version=version)
```
Aliases in practice: `@champion` (production), `@challenger` (candidate under test), `@baseline` (comparison reference), `@archived` (deprecated). Alias and version URIs are interchangeable for loading.

**Finding the latest version**:
```python
versions = mlflow.MlflowClient().search_model_versions(f"name = '{model_name}'")
latest = max(int(v.version) for v in versions)
```

**Logging a HuggingFace/transformers model (the SLM used for batch inference)**:
```python
import mlflow
signature = mlflow.models.infer_signature(
    model_input="long article text",
    model_output=mlflow.transformers.generate_signature_output(summarizer, "long article text"),
)
mlflow.transformers.log_model(
    transformers_model=summarizer,
    artifact_path="summarizer",
    task="summarization",                    # e.g. "summarization", "question-answering", "text-generation"
    inference_config={"min_length": 20, "max_length": 40, "truncation": True, "do_sample": True},
    signature=signature,
    input_example="long article text",
    registered_model_name=model_name,
)
```
`inference_config` pins the generation parameters **with** the model, so they're reapplied automatically on reload — you don't pass them again at predict time. `generate_signature_output()` produces the output half of the signature for transformers pipelines.

## 4.2 Batch inference — three patterns
```python
# A. Single-node (small/medium batches)
model = mlflow.pyfunc.load_model(model_uri=f"models:/{model_name}/{version}")
summaries = model.predict(df.limit(2).toPandas()["document"])

# B. Multi-node via spark_udf (distributed)
udf = mlflow.pyfunc.spark_udf(spark, model_uri=f"models:/{model_name}@champion",
                              env_manager="local", result_type="string")
results_df = df.withColumn("generated_summary", udf("document"))
results_df.write.mode("append").saveAsTable(f"{catalog}.{schema}.batch_results")

# C. Pure SQL via ai_query (Foundation Model API, no custom model code)
# SELECT id, ai_query('databricks-meta-llama-3-3-70b-instruct',
#        CONCAT('Summarize in <100 words: ', document)) AS summary FROM data
```
`env_manager`: `"local"` (reuse notebook env) or `"conda"` (recreate env). `result_type`: any Spark SQL type string.

## 4.3 Promote models, not code
Track lifecycle via the **Model Registry** (alias reassignment), not via code branches: log model → tag `@challenger` → evaluate against `@baseline` → promote to `@champion` → rollback = just point `@champion` at the previous version. Single source of truth, reproducible (environment captured with the model), no code redeploy needed to roll back.

## 4.4 Deploying to a Model Serving endpoint
```python
import mlflow
from mlflow.models.resources import DatabricksVectorSearchIndex, DatabricksServingEndpoint
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import EndpointCoreConfigInput, ServedEntityInput

mlflow.set_registry_uri("databricks-uc")
with mlflow.start_run():
    logged = mlflow.pyfunc.log_model(
        name="rag_chain", python_model="chain.py",
        resources=[DatabricksVectorSearchIndex(index_name=index_name),
                   DatabricksServingEndpoint(endpoint_name=llm_endpoint)],
    )
registered = mlflow.register_model(model_uri=logged.model_uri, name=f"{catalog}.{schema}.rag_chain")

w = WorkspaceClient()
w.serving_endpoints.create_and_wait(
    name=endpoint_name,
    config=EndpointCoreConfigInput(served_entities=[ServedEntityInput(
        entity_name=f"{catalog}.{schema}.rag_chain",
        entity_version=registered.version,
        workload_size="Small",
        scale_to_zero_enabled=True,
        environment_vars={"DATABRICKS_TOKEN": "{{secrets/genai_training/depl_demo_token}}"},
    )]),
)
```

**`EndpointCoreConfigInput` → `served_models` / `served_entities` full field list:**
| Field | Values / notes |
|---|---|
| `model_name` / `entity_name` | 3-level UC name of the registered model |
| `model_version` / `entity_version` | Integer version (or use an alias-based entity) |
| `workload_size` | `"Small"` / `"Medium"` / `"Large"` — concurrency tier |
| `workload_type` | CPU vs GPU tier (e.g. `"CPU"`, `"GPU_SMALL"`) for custom models |
| `scale_to_zero_enabled` | `True` = scale to zero when idle (cheap, cold-start latency); `False` = always-on (**required for PT**) |
| `environment_vars` | Dict, typically secret references |
| `served_entity_name` | **Must be unique** when >1 entity on an endpoint (A/B testing) — duplicates fail with "Served entities must have unique served entity names" |

**`WorkspaceClient.serving_endpoints` methods**: `.create_and_wait(name, config)` · `.update_config_and_wait(name=..., served_models=[...])` (modify a live endpoint, e.g. add a second entity for A/B) · `.list()` · `.get(name)` · `.query(endpoint_name, inputs=...)`. Query responses come back with a `.predictions` attribute.

**Escaping secrets inside a Python f-string**: because both the f-string and the secret syntax use braces, the literal you need is **quadruple** braces before formatting:
```python
env = {"DATABRICKS_TOKEN": "{{{{secrets/{scope}/{key}}}}}".format(scope=scope, key=key)}
# -> renders as {{secrets/my_scope/my_key}}
```

## 4.5 Provisioned Throughput (PT)
Use when you need: **throughput SLA guarantees**, compliance/data-isolation, elimination of cold starts, and predictable fixed-capacity cost (vs. variable per-token pay-per-token cost).
- PT serves foundation models from the **`system.ai`** catalog (or your own fine-tuned copies registered in UC).
- **`scale_to_zero_enabled` must be `False`** — an always-on capacity reservation is the whole point.
- Pay-per-token endpoints are the opposite trade-off: cheapest at low/spiky volume, no capacity guarantee.

## 4.6 What is the deployed artifact?
| Pattern | Artifact | Inference |
|---|---|---|
| **Registered-model pattern** | A model registered in **Unity Catalog** | The serving endpoint runs the model directly |
| **Databricks App pattern** | The **application code itself** (a DAB) | The app calls a serving endpoint for raw LLM inference; the app owns all orchestration/tool logic |

Full detail of the app pattern is in §4.10.

## 4.7 A/B testing on one endpoint
Put two served entities on the same endpoint with distinct `served_entity_name` values and split traffic between them, then attribute quality/latency metrics per entity:
```python
w.serving_endpoints.update_config_and_wait(name=endpoint_name, served_models=[...])
```
The inference table records `served_entity_name` on every row, so you slice monitoring metrics by it (`slicing_exprs=["served_entity_name"]`, §4.8). Traffic splitting can also be governed centrally by the AI Gateway (§2.13).

## 4.8 Inference tables & Lakehouse Monitoring
Enable them **after** deployment via the endpoint UI: **Configure AI Gateway → Enable inference tables**, then pick catalog / schema / table-name prefix. Tables take a few minutes to appear and only populate once the endpoint receives traffic.

Columns include `request_date`, `databricks_request_id`, `timestamp_ms`, `status_code`, `execution_time_ms`, `request`, `response`, `served_entity_name`, `sampling_fraction`, `request_metadata`.

**Payload formats** — `request` and `response` are **JSON strings**, not structs, so they must be parsed:
- `request`: the model's serving input schema, e.g. `{"inputs": [{"query": "..."}]}` or `{"messages": [...]}`.
- `response`: the model's output schema, e.g. `{"predictions": ["..."]}`.
```python
from pyspark.sql.functions import get_json_object
df = (spark.readStream.table(inference_table)          # streaming read for incremental processing
        .withColumn("input",  get_json_object("request",  "$.inputs[0].query"))
        .withColumn("output", get_json_object("response", "$.predictions[0]")))
```

**Quality metrics computed with `@pandas_udf`** over the unpacked `input`/`output` columns:
| Metric | Library | Returns |
|---|---|---|
| `compute_num_tokens` | `tiktoken` | int — token count / cost proxy |
| `compute_toxicity` | `evaluate.load("toxicity")` | double 0–1 |
| `compute_perplexity` | `evaluate.load("perplexity")` | double — fluency/confidence proxy |
| `flesch_kincaid_grade`, `automated_readability_index` | `textstat` | double — readability |

All must be **null-safe** — fill nulls before scoring (`.fillna("")`) and restore them after (`.where(texts.notna(), None)`) so empty payloads don't crash the UDF or fake a score.

```python
from delta.tables import DeltaTable
(DeltaTable.createOrReplace(spark).tableName(processed_table_name)
    .addColumns(requests_with_metrics.schema)
    .property("delta.enableChangeDataFeed", "true")   # incremental monitoring refresh
    .property("delta.columnMapping.mode", "name")     # allows column names like "toxicity(input)"
    .execute())
```

**Monitor types** (mutually exclusive — pass exactly one, or neither for Snapshot):
| Type | Class | Key params | Use for |
|---|---|---|---|
| Time series | `MonitorTimeSeries` | `timestamp_col`, `granularities` (`["5 minutes"]`, `["1 hour"]`, `["1 day"]`) | Timestamped inference logs |
| Inference log | `MonitorInferenceLog` | `timestamp_col`, `prediction_col`, `model_id_col`, `problem_type`, optional `label_col` (ground truth), `granularities` | One row per request with predictions and optionally labels |
| Snapshot | *(omit both)* | — | One-time, non-temporal analysis of a whole table |

```python
import os
from databricks.sdk.service.catalog import MonitorTimeSeries

w.quality_monitors.create(
    table_name=processed_table_name,                  # 3-level UC name
    time_series=MonitorTimeSeries(timestamp_col="timestamp", granularities=["5 minutes"]),
    assets_dir=os.getcwd(),                           # where dashboard assets are written
    slicing_exprs=["model_id", "served_entity_name"],  # break metrics down by these columns
    output_schema_name=f"{catalog}.{schema}",
)

# creation is ASYNC — poll, don't assume success:
status = w.quality_monitors.get(table_name=processed_table_name).status
# MONITOR_STATUS_PENDING -> MONITOR_STATUS_ACTIVE
```
Auto-creates **`{table}_profile_metrics`** (per granule: `metric_name`, `column_name`, `value`, `timestamp` — distributions, null counts, cardinality) and **`{table}_drift_metrics`** (per granule per column: KL divergence, anomaly flags, trend), plus a **SQL dashboard** linked from Catalog UI → Quality tab (needs a running DBSQL warehouse to render/refresh). First refresh ≈ 10–15 min. `quality_monitors.create()` can fail quietly — check the status rather than relying on an exception.

## 4.9 Environment variable configuration (deployment secrets)
```bash
databricks secrets create-scope genai_training
databricks secrets put-secret --json '{"scope":"genai_training","key":"depl_demo_token","string_value":"<PAT>"}'
```
```python
import os
token = os.environ.get("DATABRICKS_TOKEN")   # Databricks Apps read env vars this way, never hardcoded
```
Reference a secret from an endpoint/app config as `{{secrets/<scope>/<key>}}` — never a literal token in code or a committed `.env`.

## 4.10 The agent lifecycle & Databricks Apps deployment (W5)

### The four stages of the agent lifecycle on Databricks — **in order**
**Deployment → Observability → Evaluation → Monitoring**

| # | Stage | Tooling | What happens |
|:-:|---|---|---|
| 1 | **Deployment** | DABs + Databricks Apps | Package the agent and deploy it as an App with its own compute, service principal and URL |
| 2 | **Observability** | MLflow Tracing | Every request produces a structured trace: inputs, outputs, latency, execution flow |
| 3 | **Evaluation** | Scorers + judges | Scorers attached to the MLflow Experiment automatically assess trace quality |
| 4 | **Monitoring** | Backfill + archival | Manage scorer lifecycles, backfill historical traces, archive traces to Delta |

The stages are **incremental, not strictly sequential** — you can add tracing before deployment, attach scorers to a live agent, or backfill onto traces logged weeks ago. (This ordering resolves the practice question that offers permutations of *Evaluation, Deployment, Monitoring, Observability*.)

### Databricks Apps — key characteristics
- **Managed compute** — the platform provisions and manages resources; you configure no clusters, VMs or containers.
- **Service principal** — each app runs under its own identity, which determines what it can reach (serving endpoints, UC objects, experiments).
- **Unique URL** — every deployed app gets a workspace-scoped URL callable by users or downstream services.
- **Workspace integration** — apps are first-class objects on the **Apps** page: status, logs, redeploy.

In this pattern **the agent code is the deployment artifact**. The app still calls a model serving endpoint for LLM inference, but orchestration, tool integrations and observability wiring live in your application source.

### Declarative Automation Bundles (DABs)
The project folder holds two categories of file:

**User-authored source**
| File | Purpose |
|---|---|
| `agent.py` | Agent logic (e.g. OpenAI Agents SDK pointed at a Databricks serving endpoint) |
| `server.py` | FastAPI server wrapping the agent and exposing HTTP endpoints |
| `pyproject.toml` | Python dependency specification |

**Configuration (DAB / Apps spec)**
| File | Purpose |
|---|---|
| `databricks.yml` | Bundle declaration: app name, **resources** (serving endpoints, MLflow experiments), permissions |
| `app.yaml` | Runtime manifest: startup **command** and env-var injection via its **top-level `env:` section** |

⚠ **Deployment is TWO commands, not one — and they are independent:**
```bash
# Step 1 — create/update the app resource + configuration in the workspace
databricks bundle deploy

# Step 2 — push the actual application source code
databricks apps deploy my-agent-app \
  --source-code-path /Workspace/Users/<your-username>/my-agent-app
```
`bundle deploy` manages **resource declarations**; `apps deploy` pushes **source code**. Re-running `bundle deploy` does not push code. You can redeploy code without touching resources, and update resources without redeploying code.

### Model Serving endpoint vs. Databricks App
| Concern | Model Serving endpoint | Databricks App |
|---|---|---|
| What is deployed | A registered model (MLflow/UC) | Application code (DAB) |
| Inference | The endpoint runs the model directly | The app **calls** a serving endpoint |
| Scaling | Managed by serving infrastructure | Managed by the Apps platform |
| Customization | Limited to model configuration | Full control: custom UI, middleware, routing logic |
| Governance | UC model permissions | App-level ACLs **+** UC permissions on accessed resources |
| Tool integration | Defined in the registered model | MCP servers, direct API calls, any Python library |

⚠ Registering the agent model in UC is **optional** with app-based deployment. UC still governs the data and resources the app touches, but **the app itself is not a UC securable** — governance splits between workspace-level app ACLs and UC permissions on downstream resources.

### Resource injection — `databricks.yml` → `app.yaml` → `os.environ`
```yaml
# databricks.yml -- declares the app and its resources (env vars nested under config.env)
bundle:
  name: my-agent-app
resources:
  apps:
    my_agent:
      name: my-agent-app
      source_code_path: ./src
      config:
        command: ["uv", "run", "start-server"]
        env:
          - name: DATABRICKS_HOST
            value: ${workspace.host}
          - name: SERVING_ENDPOINT
            value: my-llm-endpoint
          - name: MLFLOW_EXPERIMENT_NAME
            value: /Users/me/my-experiment
```
```yaml
# app.yaml -- runtime manifest; top-level env:, valueFrom references a resource key
command: ["uv", "run", "start-server"]
env:
  - name: SERVING_ENDPOINT
    valueFrom: "my-serving-endpoint"
  - name: SECRET_KEY
    valueFrom: "my-secret"
  - name: MLFLOW_EXPERIMENT_NAME
    value: "/Users/me/my-experiment"
```
```python
# agent code reads configuration from the environment -- never hardcoded, never CLI args, never a committed .env
import os
serving_endpoint = os.environ["SERVING_ENDPOINT"]
experiment_name  = os.environ["MLFLOW_EXPERIMENT_NAME"]
workspace_host   = os.environ["DATABRICKS_HOST"]
```
**Exam points**: env vars are nested under **`config.env`** in `databricks.yml` but under a **top-level `env:`** in `app.yaml`. `valueFrom` references a resource key declared in the app's resources block (use it for endpoints and secrets); `value` is a literal. `DATABRICKS_HOST` is **automatically available** in all Databricks Apps — it only needs explicit injection when using bundle substitution. The payoff: the *same* agent code runs in dev/staging/prod by changing only `databricks.yml`.

### Governance in the deployed app
Requests from the app to serving endpoints pass through the **Unity AI Gateway** (§2.13) automatically — guardrails, rate limits, usage tracking, traffic splitting, inference tables — **with no agent code changes**. The app's **service principal** must hold the permissions for everything it touches (e.g. `EXECUTE` on UC functions reached via MCP).

---
# 5. Exam Traps & Gotchas — quick-hit review pass

Read this section last, right before the exam. Each line is a documented "trap" answer pattern.

### §0–§1 — fundamentals, parsing, chunking, retrieval
- **Context window overflow** → drops the **earliest** info, not the latest; this is a hallucination driver, not just a cost issue.
- **Lost in the middle** → info buried in the *middle* of a long context gets ignored, not the start/end.
- **LLM-as-judge biases** → verbosity bias (favours longer answers), position bias, self-preference bias.
- **Model class fit** → SLM = high-volume/low-latency; reasoning model = deep multi-step problems; frontier = hardest problems, highest cost.
- **Prompt vs context engineering** → prompt = system instructions + few-shot + user prompt. Context = all of that **plus** retrieved docs & metadata, history/constraints (Lakebase), and tools (MCP + Genie).
- **`ai_parse_document`** → pass `map('version','2.0')`; v2.0 is the recommended schema (HTML tables, better layout) and the v2 `ai_classify`/`ai_extract` patterns assume it. Always check `error_status`.
- **Pipeline order** → Parse → (Classify + Extract **in parallel with** Chunk, all reading the parsed VARIANT) → join classify/extract results onto the chunk table as filterable metadata.
- **`variant_explode`** returns exactly `pos` (INT), `key` (STRING, **NULL for arrays**), `value` (VARIANT). **NULL/non-VARIANT input yields NO rows** — use **`variant_explode_outer`** to keep the row. `posexplode` needs an already-typed ARRAY column, so it does not work on parse output.
- **`ai_classify`** → best label is `cls:response[0]`, not `cls` directly, not `cls::INT`, not a joined `cls:labels` string.
- **`ai_prep_search`** → embed `chunk_to_embed` (context-enriched), retrieve/show `chunk_to_retrieve` (clean) — reversing these is the classic wrong-answer trap.
- **Chunking numbers** → chunks must fit the embedding model's window (**e.g. 512 tokens**), overflow is **silently truncated**; keep **10–20% overlap**.
- **Embedding alignment** → the *same* embedding model must index documents and encode queries.
- **`create_delta_sync_index` params** → `endpoint_name`, `source_table_name`, `index_name`, `pipeline_type` (`TRIGGERED`/`CONTINUOUS`), `primary_key`, `embedding_source_column`, `embedding_model_endpoint_name`, **`columns_to_sync`** (the columns returned alongside results). Self-managed embeddings use `embedding_dimension` + `embedding_vector_column` instead.
- **CDF is required** on the source Delta table for standard-endpoint incremental sync — not `query_type="FULL_TEXT"`, not raising `num_results`, not moving to a Volume (indexes sync from **tables**, not Volumes).
- **AI Search `filters`** are a dict of `"column OP"` keys (`"page_number >": 10`), and string matching is on **whitespace-separated tokens**.
- **`num_results` 10–100**, and prefer the **lowest** embedding dimension that preserves quality.
- **Cosine similarity** measures the **angle** (length-insensitive); ≈1.0 = similar, →0 = dissimilar. ANN trades a little precision for large speed gains vs exact KNN.
- **Knowledge Assistant sources**: **≤10 per agent**. Volume files (`txt/pdf/md/ppt(x)/doc(x)`) with **files >50 MB automatically skipped**; an existing AI Search index **must use `databricks-gte-large-en`**; a file **table** needs streaming **OR** CDF enabled + `content` (BINARY/STRING) + `_metadata`/`metadata` struct — not Parquet-partitioning, not a Volume conversion, not a precomputed embedding column.
- **KA pipeline = 4 stages**: Indexing → Retrieval (the **Instructed Retriever** plans sub-queries) → Generation (with doc & page-level **citations**) → Feedback loop (MLflow evaluation layer + **ALHF** optimization layer).
- **Creating a KA provisions BOTH** an AI Search endpoint and a Model Serving endpoint; deleting the agent removes both.

### §2 — agents, tools, Agent Bricks, MCP, gateway
- **Single vs multi-agent** → start with a **single agent**; it holds up to roughly **8–10 tools** (heuristic). Decompose only for tool-selection errors, conflicting instructions, or independent team ownership. Multi-agent costs extra routing latency and tokens.
- **Genie space vs UC function vs AI Search vs Knowledge Assistant**: Genie = open-ended NL→SQL over structured tables; UC function = fixed, known, parameterized logic; AI Search = unstructured document retrieval; KA = the packaged document-Q&A Agent Brick.
- **Genie integrates two ways**: via a **managed MCP** endpoint (as a tool), or as a **subagent** under a Supervisor Agent.
- ⚠ **Agent Bricks types (current)**: **Knowledge Assistant, Supervisor Agent, Classification, Information Extraction**. **Custom LLM is legacy** (as is the legacy IE flow) — some workspaces still show them. The **Supervisor Agent coordinates Genie spaces, Knowledge Assistants, UC functions and external MCP servers**.
- **Agent Bricks success criteria** to state up front: **groundedness, accuracy, coverage, latency, cost**. Every brick sits on **Unity Catalog + Model Serving + MLflow Tracing + Agent Evaluation**.
- **UC/Python function tools**: explicit type hints required, **no `*args`/`**kwargs`**, imports inside the function body, **no Pydantic `BaseModel`** types.
- **`ENVIRONMENT` clause** is how a `LANGUAGE PYTHON` UC function gets third-party dependencies.
- **`@function_tool`** turns a plain Python function into a tool **without UC registration**; its schema comes from the **function name + type hints + docstring** — so vague docstrings directly cause bad tool selection. Trade-off: no UC governance/discoverability.
- **Tool names use `__` not `.`** when exposed to an LLM — agent code must `replace("__", ".")` before `execute_function`, and check `result.error` as well as `result.value`.
- **Managed MCP servers cover**: **UC functions, Genie spaces, AI Search indexes, and Databricks SQL**.
- **Managed MCP URL shape**: `/api/2.0/mcp/functions/<catalog>/<schema>` — not `/sql/`, `/genie/`, `/models/`. Transport is **Streamable HTTP**.
- **MCP governance**: UC permissions are still enforced through a managed MCP server — MCP does **not** bypass governance. The app's service principal needs `EXECUTE` on the functions.
- **MCP tools are discovered at runtime** (`tools/list`) — new UC functions registered on a live MCP server become available to a deployed agent with **no redeployment**.
- **MCP lifecycle order**: `initialize` (+ `initialized` notification) → `tools/list` → `tools/call` → shutdown. One client = one server; a host may hold many clients. Primitives = **Tools / Resources / Prompts**.
- **`McpServer.from_uc_function` must be used as an async context manager** (`async with ...`), and takes `catalog`/`schema`/`workspace_client`/`name`.
- **Orchestration patterns split by who decides**: **LLM-driven** = handoffs, agents-as-tools. **Code-driven** = sequential chaining, parallel execution, feedback loops.
- **OpenAI Agents SDK handoffs**: `handoffs=[...]` auto-generates `transfer_to_{name}` tools; after handoff `result.last_agent` is the **worker**, not the supervisor; use `result.to_input_list()` (not `result.final_output`) to keep full history for the next turn. Call `set_trace_processors([])` so MLflow owns tracing.
- **Any framework (LangChain, LangGraph, DSPy, OpenAI SDK) must be wrapped in `ResponsesAgent`** to be deployed.
- **Unity AI Gateway**: centralizes guardrails / rate limits / usage tracking / traffic splitting / inference tables / fallbacks across model **and** MCP endpoints (and coding agents), with **zero agent code changes**. Currently **Beta**, enabled from the Previews page.
  - Rate limiting is **per endpoint AND per identity** (user or group) in QPM/TPM.
  - **Fallbacks trigger on 429/5xx**, walking the configured model order; every attempt is logged.
  - **MCP governance = on-behalf-of execution**: tools run with the **requesting user's** permissions, not a shared service account.
  - In the Beta gateway, **usage tracking is on by default** once enabled.
- **Trace session grouping**: session id must be **metadata**, not a **tag**, or multi-turn/session judges won't pick it up. `mlflow.update_current_trace(session_id=...)` writes it to metadata for you.
- **UC trace tables**: binding an experiment to `trace_location=UnityCatalog(...)` creates **4** tables (`_otel_spans` / `_otel_annotations` / `_otel_logs` / `_otel_metrics`) — not one combined table. The binding is **permanent** for that experiment.
- **Trace ingestion limits**: 200 traces/sec per workspace, 100 MB/sec per table.
- **Span extras (e.g. token counts) are span attributes**, not new top-level trace fields (MLflow spans are OpenTelemetry-compatible).

### §3 — evaluation & scorers
- **Judge spectrum, least→most customizable**: Built-in judges → Guideline judges → Custom LLM judges (`make_judge()`) → Code-based scorers (`@scorer`) → **Third-party scorers**.
- ⚠ **`predict_fn` contract**: accepts the **`inputs` dict keys as keyword arguments**, returns a **JSON-serializable dict**, is **traced**, and emits **exactly one trace per call**. MLflow auto-applies `@mlflow.trace` if it isn't traced.
- **Evaluation dataset record fields**: `inputs` (required) + optional `outputs`, `expectations`, **`source`**, **`tags`**.
- **`mlflow.genai.evaluate()` params**: `data`, `scorers`, optional `predict_fn`, optional **`model_id`** (links to a LoggedModel).
- **`EvaluationResult`** exposes `run_id`, `metrics`, **`artifacts`**, **`tables`** — per-example results come from `mlflow.search_traces(run_id=..., return_type="list")` then `trace.info.assessments`, **not** a `.result_df`.
- ⚠ **Retrieval judges need traces with `RETRIEVER` spans**: `RetrievalGroundedness`, `RetrievalRelevance`, `RetrievalSufficiency`. No tracing → these cannot run at all.
- **Ground-truth-requiring judges**: `Correctness`, `RetrievalSufficiency` (+ RETRIEVER spans), `ExpectationsGuidelines`. **No ground truth**: `RelevanceToQuery`, `RetrievalRelevance`, `RetrievalGroundedness`, `Safety`, `Guidelines`, `ToolCallEfficiency`. **`ToolCallCorrectness` = optional.**
- **`Guidelines`** = same rules for every row (global). **`ExpectationsGuidelines`** = different rules per row from that row's own `expectations`, and **requires an `outputs` field**.
- **Guidelines judges return the strings `"yes"`/`"no"`**, not booleans.
- ⚠ **Write guidelines in terms of "the request" and "the response"** — the judge extracts those from the trace. Use "The response must / must not / may optionally …".
- **Feedback vs Expectation assessments**: Feedback = judgment on the actual output. Expectation = the ground truth it should have matched.
- **`Feedback` has four fields**: `value`, `rationale`, `source`, `metadata`. **`source` is auto-populated**: `CODE` for `@scorer`, `LLM_JUDGE` for `make_judge()` — set it manually only when calling your own LLM inside a `@scorer`.
- **`@scorer` may accept only** `inputs`, `outputs`, `expectations`, `trace` (any combination).
- **Answer-sheet evaluation**: omit `predict_fn` and supply an `outputs` column to score pre-generated responses.
- **`make_judge` import** is `from mlflow.genai.judges import make_judge` (also `from mlflow.genai import make_judge`) — built-in judge classes, `@scorer`, `ScorerSamplingConfig`, `list_scorers`/`get_scorer`/`delete_scorer` all come from `mlflow.genai.scorers`.
- **`make_judge()` template variables**: only `{{inputs}}`, `{{outputs}}`, `{{expectations}}`, `{{trace}}`, `{{conversation}}` — never invented ones like `{{question}}`. `{{conversation}}` combines only with `{{expectations}}`. `model=` is **required** for `{{trace}}`-based judges.
- **`feedback_value_type`** excludes Pydantic `BaseModel`.
- **`@scorer` serialization**: imports **inside** the function body, defined in a notebook, no class-based scorers, no external references, no import-requiring type hints. Use `aggregations=["mean","min","max"]` for experiment-level stats.
- ⚠ **Production monitoring supports** built-in judges, `make_judge()`, `Guidelines` and `@scorer` functions via `.register()` + `.start()`. **Only direct `Scorer` subclasses and third-party scorers are excluded.**
- **Scorer two-step**: `.register(name=...)` binds it (it does **not** start evaluating), `.start(sampling_config=...)` activates it.
- **Scorer lifecycle methods always return a NEW instance** — forgetting to reassign (`safety = safety.stop()`) leaves the old instance running.
- **Must `.stop()` before `delete_scorer()`**; historical assessments survive both stop and delete.
- **Max 20 scorers per experiment**; the background Trace Metrics Computation Job runs until **all** scorers are stopped.
- **Sampling strategy**: safety-critical scorer = 100%; expensive quality judge = ~5–10%; never both at the same rate.
- **Multi-turn assessments attach to the FIRST trace in the session**; sessions close after a **default 5-minute** inactivity buffer (`MLFLOW_ONLINE_SCORING_DEFAULT_SESSION_COMPLETION_BUFFER_SECONDS`).
- **`mlflow.search_traces()` filters**: `trace.status = 'ERROR'`, `trace.execution_time_ms > 5000` — not `trace.state = failed`, not `duration > 5s`.
- **Metric backfill** (apply new scorer to *old* traces, `databricks.agents.scorers.backfill_scorers`, returns a `job_id`, **one job per experiment** or `RESOURCE_CONFLICT`) ≠ **trace archival** (`mlflow.tracing.archival.enable_databricks_trace_archival`, append-only Delta table, auto-created schema).
- **UC trace-table grants**: `ALL_PRIVILEGES` alone is not enough — still need explicit `USE_CATALOG` + `USE_SCHEMA` + `SELECT`/`MODIFY` on the table.
- **SME reviewers need account access + `CAN_QUERY` on the endpoint + `CAN_EDIT` on the experiment** — not full workspace access.
- **`agents.deploy()`** provisions endpoint *and* Review App in one call; hand-building `EndpointCoreConfigInput` does **not** create a Review App.

### §4 — deployment & monitoring
- **Agent lifecycle order**: **Deployment → Observability → Evaluation → Monitoring** (DABs+Apps → MLflow Tracing → Scorers+Judges → Backfill+Archival). Incremental, not strictly sequential.
- **Databricks Apps deploy is two commands**: `databricks bundle deploy` (resources) then `databricks apps deploy <app-name>` (source code) — running only the first does not push code.
- **App config values come from env vars**: declared in `databricks.yml` under **`config.env`**, surfaced in `app.yaml` under a **top-level `env:`** (`valueFrom` for resource/secret keys, `value` for literals), read via `os.environ[...]`. Never hardcoded, never CLI args, never a committed `.env`. `DATABRICKS_HOST` is auto-provided.
- **The app is not a UC securable** — registering the agent model in UC is optional in the app pattern; governance = app ACLs + UC permissions on downstream resources.
- **`served_entity_name` must be unique** per served entity on an endpoint — duplicates fail deployment; it is also what the inference table records for A/B attribution.
- **`scale_to_zero_enabled`**: `True` = cheap + cold starts; **must be `False` for Provisioned Throughput**.
- **Inference table `request`/`response` are JSON strings**, not structs — parse with `get_json_object`, and use `spark.readStream.table()` for incremental processing.
- **Inference tables close the evaluation loop**: mine them for failures/edge cases → augment the eval dataset → re-evaluate.
- **Monitor creation is async and can fail silently** — poll `quality_monitors.get(...).status` for `MONITOR_STATUS_ACTIVE`; don't rely on an exception.
- **`delta.columnMapping.mode = "name"`** is needed on the processed metrics table because metric columns are named like `toxicity(input)`; `delta.enableChangeDataFeed` is needed for incremental refresh.
- **`inference_config`** on `mlflow.transformers.log_model` pins generation params with the model — reapplied automatically on reload.
- **`DA` object** (Databricks Academy lab notebooks only, not a product feature): a training helper exposing environment-specific variables (username, catalog, schema, working dir, dataset paths) — recognize it if a lab-setup question appears.

---
# 6. Full API / Function Alphabetical Index

| Name | What it is | Section |
|---|---|---|
| `agents.deploy()` | One-call deploy of a registered agent + Review App | §3.6 |
| `Agent`, `Runner.run()`, `handoffs=`, `mcp_servers=` (OpenAI Agents SDK) | Multi-agent supervisor/worker pattern, auto-generated transfer tools, MCP wiring | §2.11, §2.12 |
| `AgentExecutor` (LangChain) | Orchestrates a tool-calling agent's reasoning loop | §2.4 |
| Agent Bricks types (Knowledge Assistant, Supervisor Agent, Classification, Information Extraction) | Declarative agent framework; Custom LLM is **legacy** | §2.10 |
| AI Bridge packages (`databricks-langchain`/`-openai`/`-dspy`) | Framework integration layer for Databricks resources | §2.0 |
| AI Playground ("Try in Playground", "Get code → Export to Databricks Apps") | Test an index or UC tools, then scaffold an app project | §1.14, §2.10 |
| `ai_classify(text, array(labels))` | Classify text into fixed labels; top label = `response[0]` | §1.8 |
| `ai_extract(text, spec)` | Extract structured fields from text as JSON | §1.9 |
| `ai_parse_document(content, map('version','2.0', ...))` | Layout-aware document parsing (OCR + multimodal LLM) → VARIANT | §1.6 |
| `ai_prep_search(...)` | Purpose-built semantic chunking; emits `chunk_to_embed` + `chunk_to_retrieve` | §1.11 |
| `ai_query(endpoint, prompt)` | Call a Foundation Model endpoint from SQL/Python; batch inference | §1.10, §4.2 |
| `app.yaml` (`command`, top-level `env:`, `valueFrom`) | Databricks App runtime manifest and env-var injection | §4.10 |
| `AssessmentSource` (`CODE` / `LLM_JUDGE`) | Auto-populated `Feedback.source` identifying who produced an assessment | §3.6 |
| `backfill_scorers` / `BackfillScorerConfig` (`databricks.agents.scorers`) | Retroactively score historical traces with a new scorer; returns `job_id` | §3.11 |
| `ChatDatabricks` (LangChain) | LLM wrapper for a Databricks Model Serving endpoint | §2.4 |
| `columns_to_sync` | Index parameter: columns stored with vectors and returned in results | §1.14 |
| `ConversationCompleteness`, `ConversationalGuidelines`, `ConversationalRoleAdherence`, `ConversationalSafety`, `ConversationalToolCallEfficiency`, `KnowledgeRetention`, `UserFrustration` | The seven built-in multi-turn judges | §3.4 |
| `Correctness`, `RelevanceToQuery`, `RetrievalSufficiency`, `RetrievalRelevance`, `RetrievalGroundedness`, `Safety`, `ToolCallEfficiency`, `ToolCallCorrectness` | Built-in single-turn MLflow judges | §3.3 |
| `create_delta_sync_index()` / `..._and_wait()` | Create an auto-syncing AI Search index from a Delta table | §1.14 |
| `create_python_function()` / `execute_function()` → `.value` / `.error` | Register/run a Python UC function tool | §2.3 |
| `create_retrieval_chain` / `create_stuff_documents_chain` | LangChain RAG chain assembly (retriever + doc-stuffing prompt) | §1.14 |
| `create_tool_calling_agent()` / `create_agent()` (LangChain) | Build a tool-using agent from an LLM + tools + prompt | §2.4 |
| `databricks bundle deploy` / `databricks apps deploy <app>` | Two-step App deployment: resources, then source code | §4.10 |
| `databricks.yml` (`resources.apps.*.config.env`) | DAB bundle declaration: app name, resources, permissions, env vars | §4.10 |
| `DatabricksMCPClient` / `McpServer.from_uc_function()` | Discover/build MCP tool access to UC functions (async context manager) | §2.11 |
| `DatabricksReranker` | Cross-encoder reranking of AI Search results | §1.13 |
| `DatabricksVectorSearch` / `.as_retriever(search_kwargs={"k":N})` | LangChain wrapper turning an index into a retriever | §1.14 |
| `delete_scorer(name=...)` | Permanently remove a registered scorer (must be stopped first) | §3.9 |
| `delta.enableChangeDataFeed` | Table property required for Delta Sync indexes / KA file tables / incremental monitoring | §1.14, §1.16, §4.8 |
| DSPy (Signatures / Modules / Compiler / Program) | Programmatic prompt-optimization framework | §2.6 |
| `enable_databricks_trace_archival` / `disable_...` (`mlflow.tracing.archival`) | Stream production traces to a UC Delta table (append-only) | §3.11 |
| `EndpointCoreConfigInput` / `served_entity_name` / `workload_size` / `workload_type` | Model Serving endpoint config schema | §4.4 |
| `ENVIRONMENT` clause (`LANGUAGE PYTHON`) | Declares third-party deps for a Python UC function | §2.3 |
| `EvaluationResult` (`run_id`, `metrics`, `artifacts`, `tables`) | Return object of `mlflow.genai.evaluate()` | §3.2 |
| `ExpectationsGuidelines` | Per-row guideline judge; requires an `outputs` field | §3.3 |
| `Feedback(value, rationale, source, metadata)` | Structured scorer output object | §3.6 |
| `@function_tool` (OpenAI Agents SDK) | Turn a Python function into a tool without UC registration | §2.3 |
| Genie space / Genie agent | NL→SQL interface over governed data; usable via managed MCP or as a subagent | §2.2, §2.10 |
| `get_json_object()` + `spark.readStream.table()` | Unpack JSON inference-table payloads incrementally | §4.8 |
| `get_scorer(name=...)` / `list_scorers()` | Fetch or audit registered scorers (incl. UI-created) | §3.9 |
| `Guidelines` | Uniform natural-language rule judge; returns `"yes"`/`"no"` | §3.3 |
| `make_judge()` | Build a custom LLM judge (prompt- or trace-based) | §3.5 |
| MCP (`/api/2.0/mcp/functions/<catalog>/<schema>`, `tools/list`, `tools/call`, Streamable HTTP) | Governed, standardized tool-exposure protocol | §2.11 |
| `mlflow.genai.evaluate(data=, scorers=, predict_fn=, model_id=)` | Run scorers over an evaluation dataset / predict function | §3.2 |
| `mlflow.genai.labeling.get_review_app()` / `.add_agent()` | SME feedback Review App | §3.6 |
| `mlflow.langchain.autolog()` / `mlflow.openai.autolog()` | Automatic tracing for LangChain / OpenAI SDK agents | §1.15, §2.7 |
| `mlflow.models.set_model()` | Marks which object in `agent.py` is the servable model | §1.15 |
| `mlflow.pyfunc.load_model()` / `spark_udf()` | Load a registered model for single-node / distributed batch inference | §4.2 |
| `mlflow.register_model()` / `set_registered_model_alias()` | Register a model to UC / assign a lifecycle alias | §1.15, §4.1 |
| `mlflow.search_traces(filter_string=, run_id=, locations=, return_type=)` | Query traces by filter expression, run or experiment | §3.10 |
| `mlflow.set_experiment(trace_location=UnityCatalog(...))` | Store OTel traces natively in 4 UC Delta tables | §2.7 |
| `mlflow.trace` (decorator) / `start_span()` / `start_active_span()` | Manual span instrumentation | §2.7 |
| `mlflow.transformers.log_model()` / `generate_signature_output()` / `inference_config` | Log a HuggingFace pipeline with pinned generation params | §4.1 |
| `mlflow.update_current_trace(session_id=, user=, metadata=, tags=)` | Set session metadata (required for multi-turn judges) and tags | §2.7, §3.4 |
| `MonitorTimeSeries` / `MonitorInferenceLog` / Snapshot (`quality_monitors.create`) | Lakehouse Monitoring monitor types and their params | §4.8 |
| `@pandas_udf` quality metrics (tiktoken / evaluate / textstat) | Toxicity, perplexity, token count, readability on payloads | §4.8 |
| `predict_fn` contract | Keyword args from `inputs`, JSON-serializable dict out, traced, one trace per call | §3.2 |
| `prep_msgs_for_cc_llm()` / `predict_stream()` | ResponsesAgent message conversion and streaming | §2.8 |
| `ResponsesAgent` | Production agent interface (OpenAI Responses-schema compatible); required for deployment | §2.8 |
| `scorer.register()` / `.start()` / `.stop()` / `.update()` | Production scorer lifecycle — **each returns a new instance** | §3.9 |
| `@scorer` decorator (`inputs`, `outputs`, `expectations`, `trace`; `aggregations=[...]`) | Code-based deterministic scorer function | §3.5 |
| `ScorerSamplingConfig(sample_rate=...)` / `ScorerScheduleConfig` | Configure what fraction of production traffic a scorer evaluates | §3.9, §3.5 |
| `search_model_versions("name = '...'")` | Enumerate registered model versions | §4.1 |
| `SentenceSplitter` / `set_global_tokenizer` (llama_index) | Token-aware alternative chunker | §1.14 |
| `serving_endpoints.create_and_wait()` / `.update_config_and_wait()` / `.query()` | Create an endpoint; add/modify served entities (A/B); invoke | §4.4 |
| `set_trace_processors([])` (OpenAI Agents SDK) | Disable the SDK's own tracing so MLflow owns it | §2.12 |
| Span types (`AGENT`, `CHAT_MODEL`, `TOOL`, `RETRIEVER`, `CHAIN`, `LLM`, `EMBEDDING`, `GUARDRAIL`, `PARSER`, `RERANKER`, `EVALUATOR`, `MEMORY`, `TASK`, `WORKFLOW`, `UNKNOWN`) | MLflow trace span classification | §2.7 |
| `UCFunctionToolkit` (LangChain / OpenAI SDK) | Wrap UC functions as callable agent tools | §2.4, §2.5 |
| Unity AI Gateway | Guardrails / rate limits (per endpoint **and** identity) / usage tracking / traffic splitting / inference tables / fallbacks (429–5xx) / MCP on-behalf-of | §2.13 |
| `variant_explode()` / `variant_explode_outer()` | Explode a VARIANT array/object into rows (`pos`, `key`, `value`); `_outer` preserves NULL rows | §1.7 |
| `VectorSearchClient` / `index.similarity_search()` / `.sync()` / `.describe()` | AI Search SDK: create/query/sync a vector index (ANN/hybrid/full-text) | §1.14 |
| `VectorSearchRetrieverTool` (LangChain) | Turns an AI Search index into an agent retriever tool | §2.4 |